In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

raw = pd.read_csv("../data_raw/raw_postings.csv", dtype=str, keep_default_na=False)
print(raw.shape)
print(raw["job_id"].is_unique, raw["job_id"].astype(int).min(), raw["job_id"].astype(int).max())
print(raw["source"].value_counts())

In [ ]:
placeholders = ["Not mentioned", "Not disclosed", "No tags", "Confidential", ""]
summary = pd.DataFrame({"n_unique": raw.nunique()})
for p in placeholders:
    summary[p or "(empty)"] = (raw == p).sum()
print(summary)

for c in ["work_mode_portal", "experience_portal", "posted_ago", "industry_type"]:
    print("\n", c)
    print(raw[c].value_counts().head(15))

In [ ]:
import re
df = raw.copy()

text_cols = ["job_title","company","city","work_mode_portal","experience_portal",
             "salary_portal","skills_portal","posted_ago","applicants",
             "industry_type","education","openings"]

def tidy(s):
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", s)
    return re.sub(r"\s+", " ", s).strip()

changed = {}
for c in text_cols:
    before = df[c].copy()
    df[c] = df[c].map(tidy)
    changed[c] = int((before != df[c]).sum())
print("values changed by trimming/whitespace:", changed)

# placeholder casing: map any case variant to the canonical form
canon = {p.lower(): p for p in ["Not mentioned","Not disclosed","No tags","Confidential"]}
for c in text_cols:
    before = df[c].copy()
    df[c] = df[c].map(lambda v: canon.get(v.lower(), v))
    n = int((before != df[c]).sum())
    if n: print("placeholder casing fixed in", c, ":", n)

df["work_mode_clean"] = df["work_mode_portal"].replace({"Hybrid work": "Hybrid"})
print(df["work_mode_clean"].value_counts())

In [ ]:
import re

d = df["description_text"]
steps = {}

def apply(name, func):
    global d
    new = d.map(func)
    steps[name] = int((new != d).sum())
    d = new

moji = {"â€™":"'", "â€˜":"'", "â€œ":'"', "â€\x9d":'"', "â€“":"-", "â€”":"-", "â€¦":"...", "Â ":" ", "Â":""}
def fix_moji(s):
    for k, v in moji.items():
        s = s.replace(k, v)
    return s

apply("mojibake", fix_moji)
apply("non-breaking hyphen", lambda s: s.replace("\u2011", "-"))
apply("non-breaking space", lambda s: s.replace("\u00a0", " "))
apply("zero-width chars", lambda s: re.sub(r"[\u200b\u200c\u200d\ufeff]", "", s))
apply("citation markers [n]", lambda s: re.sub(r"\[\d{1,2}\]", "", s))
apply("repeated spaces/tabs", lambda s: re.sub(r"[ \t]{2,}", " ", s))

df["description_clean"] = d.str.strip()
print(steps)
print("mojibake left:", int(df["description_clean"].str.contains("â€|Ã").sum()))
print("rows where description_clean != description_text:",
      int((df["description_clean"] != df["description_text"]).sum()))

In [ ]:
m = df["description_clean"].str.contains("â€|Ã")
for jid, txt in zip(df.loc[m, "job_id"], df.loc[m, "description_clean"]):
    for hit in re.finditer("â€|Ã", txt):
        s = max(0, hit.start() - 12)
        print(jid, repr(txt[s:hit.end() + 12]))

In [ ]:
fix = df["description_clean"].str.contains("â€", regex=False)
print("rows with lone â€ before fix:", int(fix.sum()))
df["description_clean"] = df["description_clean"].str.replace("â€", "-", regex=False)

left = df["description_clean"].str.contains("â€|Ã")
print("mojibake left:", int(left.sum()))
print("rows where description_clean != description_text:",
      int((df["description_clean"] != df["description_text"]).sum()))

In [ ]:
markers = {
    "LinkedIn: Requirements added by the job poster": "Requirements added by the job poster",
    "LinkedIn: Benefits found in job post": "Benefits found in job post",
    "Indeed: Application Question": "Application Question",
    "Powered by JazzHR": "Powered by JazzHR",
    "Scraped metadata: Funding Stage": "Funding Stage",
    "Scraped metadata: Business Type": "Business Type",
    "Disclaimer": "Disclaimer",
    "Equal opportunity": "equal opportunity",
    "Fraud warning": "fraud",
}
for label, pat in markers.items():
    m = df["description_clean"].str.contains(pat, case=False, regex=False)
    print(f"{label}: {int(m.sum())} rows | by source: {df.loc[m,'source'].value_counts().to_dict()}")
    print("   job_ids:", df.loc[m, "job_id"].tolist()[:25])

In [ ]:
checks = {
    "poster": "Requirements added by the job poster",
    "benefits": "Benefits found in job post",
    "appq": "Application Question",
    "jazzhr": "Powered by JazzHR",
    "funding": "Funding Stage",
}
for label, pat in checks.items():
    print("\n==", label)
    for jid, txt in zip(df["job_id"], df["description_clean"]):
        i = txt.lower().find(pat.lower())
        if i >= 0:
            pct = round(100 * i / len(txt))
            print(jid, f"at {pct}% of {len(txt)} chars:", repr(txt[i:i+60]))

print("\n== disclaimer (first 8 rows)")
n = 0
for jid, txt in zip(df["job_id"], df["description_clean"]):
    i = txt.lower().find("disclaimer")
    if i >= 0 and n < 8:
        n += 1
        print(jid, f"at {round(100*i/len(txt))}% of {len(txt)} chars:", repr(txt[i:i+60]))

In [ ]:
def cut_to_end(s, marker):
    i = s.find(marker)
    return s[:i].rstrip() if i >= 0 else s

for marker in ["Requirements added by the job poster",
               "Benefits found in job post",
               "Powered by JazzHR",
               "Application Question(s):"]:
    before = df["description_clean"]
    after = before.map(lambda s, m=marker: cut_to_end(s, m))
    print(marker, "| rows cut:", int((before != after).sum()),
          "| chars removed:", int((before.str.len() - after.str.len()).sum()))
    df["description_clean"] = after

print("\n== metadata rows: first 450 chars")
for jid in ["111", "133", "147"]:
    t = df.loc[df["job_id"] == jid, "description_clean"].iloc[0]
    print(jid, repr(t[:450]))

print("\n== disclaimer: 420 chars from the marker")
for jid in ["45", "53", "55", "59", "61"]:
    t = df.loc[df["job_id"] == jid, "description_clean"].iloc[0]
    i = t.lower().find("disclaimer")
    print(jid, repr(t[i:i+420]))

In [ ]:
def strip_meta(s):
    if s.startswith("Company:") and "Job Description" in s:
        i = s.find("Job Description")
        return s[i + len("Job Description"):].lstrip()
    return s

before = df["description_clean"]
after = before.map(strip_meta)
print("metadata blocks cut:", int((before != after).sum()),
      "| chars removed:", int((before.str.len() - after.str.len()).sum()))
df["description_clean"] = after

print("\n== structure after the disclaimer")
for jid in ["45", "59", "30"]:
    t = df.loc[df["job_id"] == jid, "description_clean"].iloc[0]
    i = t.lower().find("disclaimer")
    print("\njob", jid, "| text after disclaimer:", len(t) - i, "chars")
    for line in t[i:].split("\n")[1:]:
        if line.strip():
            print("  ", repr(line[:40]))

print("\n== do disclaimer rows have real content after it?")
kw = ["sql", "excel", "power bi", "experience", "responsib"]
for jid, t in zip(df["job_id"], df["description_clean"]):
    i = t.lower().find("disclaimer")
    if i >= 0:
        tail = t[i:].lower()
        hits = [k for k in kw if k in tail]
        if hits:
            print(jid, hits)

In [ ]:
def cut_disclaimer(s):
    i = s.lower().find("disclaimer")
    return s[:i].rstrip() if i >= 0 else s

before = df["description_clean"]
after = before.map(cut_disclaimer)
cut = before != after
print("disclaimer blocks cut:", int(cut.sum()),
      "| by source:", df.loc[cut, "source"].value_counts().to_dict(),
      "| chars removed:", int((before.str.len() - after.str.len()).sum()))
df["description_clean"] = after

# same Naukri field block in rows that had no disclaimer?
m = df["description_clean"].str.contains("Role Category:", regex=False)
print("\nrows still containing 'Role Category:':", int(m.sum()),
      "| by source:", df.loc[m, "source"].value_counts().to_dict())
print("job_ids:", df.loc[m, "job_id"].tolist()[:30])

In [ ]:
from collections import Counter

m = df["description_clean"].str.contains("Role Category:", regex=False)
heads = Counter()
pcts, tails, odd = [], [], []

for jid, t in zip(df.loc[m, "job_id"], df.loc[m, "description_clean"]):
    rc = t.find("Role Category:")
    start = t.rfind("\nRole:", 0, rc)
    if start < 0:
        odd.append(jid)
        continue
    tail = t[start:]
    pcts.append(round(100 * start / len(t)))
    tails.append(len(tail))
    for line in tail.split("\n"):
        line = line.strip()
        if line:
            heads[line.split(":")[0][:30]] += 1

print("rows with a block start found:", len(pcts), "| no start found:", odd)
print("block start as % of text: min", min(pcts), "median", sorted(pcts)[len(pcts)//2], "max", max(pcts))
print("block length chars: min", min(tails), "max", max(tails))
print("\nmost common line labels inside the block + after it:")
for k, v in heads.most_common(25):
    print(f"  {v:3d}  {k!r}")

In [ ]:
m = df["description_clean"].str.contains("Role Category:", regex=False)
overlap = df.loc[cut & m, "job_id"].tolist()
print("rows cut for disclaimer that still had the block:", overlap)

def cut_field_block(s):
    rc = s.find("Role Category:")
    if rc < 0:
        return s
    start = s.rfind("\nRole:", 0, rc)
    return s[:start].rstrip() if start >= 0 else s

before = df["description_clean"]
after = before.map(cut_field_block)
c2 = before != after
print("\nfield blocks cut:", int(c2.sum()), "| by source:", df.loc[c2, "source"].value_counts().to_dict(),
      "| chars removed:", int((before.str.len() - after.str.len()).sum()))
df["description_clean"] = after

print("\n== rows 3 and 67: 250 chars around the marker")
for jid in ["3", "67"]:
    t = df.loc[df["job_id"] == jid, "description_clean"].iloc[0]
    i = t.find("Role Category:")
    print(jid, f"at {round(100*i/len(t))}% of {len(t)} chars:", repr(t[max(0, i-120):i+130]))

print("\nrows still containing 'Role Category:':", int(df["description_clean"].str.contains("Role Category:", regex=False).sum()))

In [ ]:
t = df.loc[df["job_id"] == "3", "description_clean"].iloc[0]
print(repr(t))

In [ ]:
def cut_inline_block(s):
    rc = s.find("Role Category:")
    if rc < 0:
        return s
    start = s.rfind(" Role:", 0, rc)
    return s[:start].rstrip() if start >= 0 else s

mask = df["job_id"] == "3"
before = df.loc[mask, "description_clean"].iloc[0]
df.loc[mask, "description_clean"] = df.loc[mask, "description_clean"].map(cut_inline_block)
print("job 3 length:", len(before), "->", len(df.loc[mask, "description_clean"].iloc[0]))
left = df.loc[df["description_clean"].str.contains("Role Category:", regex=False), "job_id"].tolist()
print("rows still containing 'Role Category:':", left)

df["desc_len"] = df["description_clean"].str.len()
print("length percentiles:", df["desc_len"].describe()[["min", "25%", "50%", "75%", "max"]].round().to_dict())

for cutoff in [200, 300, 500]:
    short = df[df["desc_len"] < cutoff]
    print("under", cutoff, "chars:", len(short), "rows | by source:", short["source"].value_counts().to_dict())
    print("   job_ids:", short["job_id"].tolist())

In [ ]:
df["thin_description"] = df["desc_len"] < 500
print("thin_description:", int(df["thin_description"].sum()))

import os
os.makedirs("../data_clean", exist_ok=True)
df.to_csv("../data_clean/stage3_descriptions.csv", index=False, encoding="utf-8")
chk = pd.read_csv("../data_clean/stage3_descriptions.csv", dtype=str, keep_default_na=False)
print(chk.shape, chk["job_id"].is_unique)

In [ ]:
title_hit = df["job_title"].str.contains(r"\bintern(?:s|ship|ships)?\b", case=False, regex=True)
df["is_internship"] = title_hit

print("is_internship by title:", int(title_hit.sum()), "| by source:", df.loc[title_hit, "source"].value_counts().to_dict())
for jid, src, t in zip(df.loc[title_hit, "job_id"], df.loc[title_hit, "source"], df.loc[title_hit, "job_title"]):
    print(jid, src, "|", t)

desc_hit = df["description_clean"].str.contains(r"\binternship\b", case=False, regex=True) & ~title_hit
print("title does not say intern but description says internship:", int(desc_hit.sum()))
for jid, src, t in zip(df.loc[desc_hit, "job_id"], df.loc[desc_hit, "source"], df.loc[desc_hit, "job_title"]):
    print(jid, src, "|", t)

In [ ]:
import re

emp = df["description_text"].str.extract(r"Employment Type:\s*(.*)")[0].str.strip()
nk = df["source"] == "Naukri"
print("Naukri Employment Type values:")
print(emp[nk].value_counts(dropna=False).to_dict())

print("Employment Type on job 97:", emp[df["job_id"] == "97"].iloc[0])
print("Naukri rows with Internship in Employment Type:", df.loc[nk & emp.str.contains("intern", case=False, na=False), "job_id"].tolist())

for jid in ["22", "38", "39", "93", "227", "97"]:
    t = df.loc[df["job_id"] == jid, "description_clean"].iloc[0]
    for m in re.finditer(r"intern", t, flags=re.I):
        s = max(0, m.start() - 70)
        print(jid, repr(t[s:m.end() + 70]))
        break

In [ ]:
title_hit = df["job_title"].str.contains(r"\bintern(?:s|ship|ships)?\b", case=False, regex=True)
df["is_internship"] = (title_hit & (df["job_id"] != "97")) | (df["job_id"] == "39")

print("is_internship:", int(df["is_internship"].sum()), "| by source:", df.loc[df["is_internship"], "source"].value_counts().to_dict())
print("job_ids:", df.loc[df["is_internship"], "job_id"].tolist())

In [ ]:
import re

remote_portal = df["work_mode_clean"] == "Remote"
print("portal says Remote:", int(remote_portal.sum()), df.loc[remote_portal, "job_id"].tolist())

pat_remote = r"(fully remote|100% remote|work from home|remote role|remote position|work mode: remote|workplace type: remote|remote across india|\bwfh\b|remote / bangalore|hybrid remote)"
print("--- remote in description, portal does not say Remote")
for jid, t, r in zip(df["job_id"], df["description_clean"], remote_portal):
    if r:
        continue
    m = re.search(pat_remote, t, flags=re.I)
    if m:
        s = max(0, m.start() - 40)
        print(jid, repr(t[s:m.end() + 40]))

emp_raw = df["description_text"].str.extract(r"Employment Type:\s*(.*)")[0].fillna("")
nk_contract = emp_raw.str.contains("contract|temporary", case=False)
print("--- Naukri Employment Type contractual/temporary:", df.loc[nk_contract, "job_id"].tolist())

pat_contract = r"(contract basis|contract role|contract position|contract to hire|contract-to-hire|contractual|fixed[- ]term|third party payroll|freelance|\bcontract\b)"
print("--- contract keywords in description")
for jid, t in zip(df["job_id"], df["description_clean"]):
    m = re.search(pat_contract, t, flags=re.I)
    if m:
        s = max(0, m.start() - 40)
        print(jid, repr(t[s:m.end() + 40]))

In [ ]:
remote_portal = df["work_mode_clean"] == "Remote"
remote_desc = df["job_id"].isin(["77", "184", "255", "264", "277", "296"])
df["is_remote"] = remote_portal | remote_desc
df["is_contract"] = df["job_id"].isin(["20", "54", "208"])

print("is_remote:", int(df["is_remote"].sum()), "| by source:", df.loc[df["is_remote"], "source"].value_counts().to_dict())
print("   from portal:", int(remote_portal.sum()), "| added from description:", int((remote_desc & ~remote_portal).sum()))
print("is_contract:", int(df["is_contract"].sum()), df.loc[df["is_contract"], "job_id"].tolist())

In [ ]:
import re

pat_bond = r"(service agreement|service bond|training bond|\bbond\b|payback|pay back|lock-in|lock in|minimum commitment|\d+(?:\.\d+)?[- ]years? (?:agreement|commitment|bond))"
print("--- bond / service agreement candidates")
for jid, t in zip(df["job_id"], df["description_clean"]):
    m = re.search(pat_bond, t, flags=re.I)
    if m:
        s = max(0, m.start() - 40)
        print(jid, repr(t[s:m.end() + 50]))

pat_elig = r"(pass[- ]?outs?|batch of 20|batch 20|persons? with disabilit|\bpwd\b|udid|tier[- ]?1|\biit\b|\bnit\b|premier institute|top[- ]tier|only .{0,25}eligible|minimum \d\d% |\d\d% (?:and above|throughout|in 10th)|first class|female candidates|women (?:only|candidates)|graduat\w+ (?:in|year|of|before) 20\d\d)"
print("--- eligibility candidates")
for jid, t in zip(df["job_id"], df["description_clean"]):
    m = re.search(pat_elig, t, flags=re.I)
    if m:
        s = max(0, m.start() - 40)
        print(jid, repr(t[s:m.end() + 50]))

pat_co = r"(staffing|recruit|manpower|placement|hr services|client of|undisclosed|reputed|freelanc|executive search|talent|consultants|outsourcing|\bmnc\b|people, jobs)"
print("--- staffing/aggregator by company name")
m = df["company"].str.contains(pat_co, case=False, regex=True)
for jid, src, co in zip(df.loc[m, "job_id"], df.loc[m, "source"], df.loc[m, "company"]):
    print(jid, src, "|", co)

pat_desc = r"(our client|on behalf of|one of our clients|hiring for our client|leading mnc|third party payroll|hiring partner)"
print("--- staffing/aggregator by description phrase")
for jid, t in zip(df["job_id"], df["description_clean"]):
    m = re.search(pat_desc, t, flags=re.I)
    if m:
        s = max(0, m.start() - 40)
        print(jid, repr(t[s:m.end() + 40]))

In [ ]:
bond = {"61": "1-year service commitment", "93": "2.5-year service agreement"}
elig = {"38": "2026 pass-outs only", "74": "top-tier institute degree",
        "88": "B.Tech from premier institute", "93": "graduating 2025 or 2026",
        "98": "B.Tech from Tier 1 college", "104": "top-tier institute (Master's/MBA)",
        "117": "60% academic cutoff, 2025-2026 batch", "255": "IIT/NIT students",
        "272": "persons with disabilities only", "290": "Tier 1 institute"}
staff = {"16": "hiring on behalf of client", "32": "hiring on behalf of client",
         "27": "recruitment firm", "28": "recruitment firm",
         "37": "employer unnamed (MNC)", "43": "employer unnamed", "56": "employer unnamed",
         "78": "outsourcing firm", "94": "client of consultancy", "96": "employer undisclosed",
         "156": "aggregator", "208": "contract via third-party payroll",
         "264": "hiring partner, employer unnamed", "267": "executive search firm",
         "300": "staffing firm", "302": "individual freelancer"}

df["has_bond_or_service_agreement"] = df["job_id"].isin(bond)
df["eligibility_restricted"] = df["job_id"].isin(elig)
df["is_staffing_or_aggregator"] = df["job_id"].isin(staff)
df["eligibility_reason"] = df["job_id"].map(elig).fillna("")
df["staffing_reason"] = df["job_id"].map(staff).fillna("")
print("bond:", int(df["has_bond_or_service_agreement"].sum()),
      "| eligibility_restricted:", int(df["eligibility_restricted"].sum()),
      "| staffing_or_aggregator:", int(df["is_staffing_or_aggregator"].sum()))

print("--- job 91 and 163 context")
for jid, pat in [("91", r"agreement|graduat"), ("163", r"tier I|MBA")]:
    t = df.loc[df["job_id"] == jid, "description_clean"].iloc[0]
    for m in re.finditer(pat, t, flags=re.I):
        s = max(0, m.start() - 90)
        print(jid, repr(t[s:m.end() + 90]))
        break

print("--- companies")
for jid in ["41", "52", "71", "84", "103"]:
    print(jid, df.loc[df["job_id"] == jid, "company"].iloc[0])

In [ ]:
bond["91"] = "2-year agreement"
elig["91"] = "bachelor's completed in 2023 or earlier"
staff["41"] = "HR firm, placeholder posting for a client"
staff["84"] = "HR firm, placeholder posting for a client"

df["has_bond_or_service_agreement"] = df["job_id"].isin(bond)
df["eligibility_restricted"] = df["job_id"].isin(elig)
df["is_staffing_or_aggregator"] = df["job_id"].isin(staff)
df["eligibility_reason"] = df["job_id"].map(elig).fillna("")
df["staffing_reason"] = df["job_id"].map(staff).fillna("")
print("bond:", int(df["has_bond_or_service_agreement"].sum()),
      "| eligibility_restricted:", int(df["eligibility_restricted"].sum()),
      "| staffing_or_aggregator:", int(df["is_staffing_or_aggregator"].sum()))

t = df.loc[df["job_id"] == "163", "description_clean"].iloc[0]
i = t.find("MBA from a tier")
print(repr(t[max(0, i - 220): i + 160]))

In [ ]:
notes_exclude = ["10","12","17","19","20","41","54","59","71","72","76","78","84","90","91",
                 "104","108","120","149","156","161","165","168","186","221","233","246",
                 "253","269","273","281"]
print("--- notes exclude list:", len(notes_exclude), "rows")
for jid in notes_exclude:
    r = df[df["job_id"] == jid].iloc[0]
    print(jid, r["source"], "|", r["job_title"][:60], "|", r["company"][:30])

pat_off = (r"(developer|engineer|accountant|accounts|trainer|teacher|tutor|sales executive|"
           r"customer (?:support|success|service)|tester|testing|scientist|\bml\b|machine learning|"
           r"\bfinance\b|financial analyst|\btax\b|audit|compliance|recruit|\bhr\b|content writer)")
m = df["job_title"].str.contains(pat_off, case=False, regex=True) & ~df["job_id"].isin(notes_exclude)
print("--- other titles matching the rule check:", int(m.sum()))
for jid, src, t, co in zip(df.loc[m, "job_id"], df.loc[m, "source"], df.loc[m, "job_title"], df.loc[m, "company"]):
    print(jid, src, "|", t[:60], "|", co[:30])

In [ ]:
reasons = {
 "teaching_or_training": ["10", "233"],
 "not_an_employer_posting": ["273"],
 "finance_or_accounting": ["12", "17", "108", "156", "165", "168", "253", "269"],
 "ml_or_ai_model_work": ["19", "20", "104", "161", "221"],
 "data_engineering": ["91", "120", "149", "246"],
 "software_dev_or_testing": ["59", "72", "76", "186"],
 "support_or_data_entry_or_placeholder": ["41", "71", "78", "84"],
 "other_job_family": ["54", "90", "281"],
}
code = {jid: r for r, ids in reasons.items() for jid in ids}
df["off_target"] = df["job_id"].isin(code)
df["off_target_reason"] = df["job_id"].map(code).fillna("")
print("off_target:", int(df["off_target"].sum()))
print(df.loc[df["off_target"], "off_target_reason"].value_counts().to_dict())

print("--- titles that say contract / fixed term / temporary")
mt = df["job_title"].str.contains(r"contract|fixed[- ]?term|temporary|freelance", case=False, regex=True)
for jid, t in zip(df.loc[mt, "job_id"], df.loc[mt, "job_title"]):
    print(jid, "|", t[:80])

print("--- 183 and 276, first 350 chars")
for jid in ["183", "276"]:
    t = df.loc[df["job_id"] == jid, "description_clean"].iloc[0]
    print(jid, repr(t[:350]))

In [ ]:
df["is_contract"] = df["job_id"].isin(["20", "54", "208", "12"])
print("is_contract:", int(df["is_contract"].sum()), df.loc[df["is_contract"], "job_id"].tolist())

import re
for jid in ["183", "276"]:
    t = df.loc[df["job_id"] == jid, "description_clean"].iloc[0]
    print("\njob", jid, "| length", len(t))
    for pat in [r"responsibilit", r"what you.{0,12}(do|bring)", r"requirements|qualifications|must have"]:
        m = re.search(pat, t, flags=re.I)
        if m:
            print("  >>", pat, "at", round(100 * m.start() / len(t)), "%:", repr(t[m.start(): m.start() + 450]))
    tools = ["sql", "excel", "power bi", "tableau", "python", "etl", "pipeline", "spark", "azure", "machine learning", "dashboard"]
    print("  tool words:", {k: len(re.findall(r"\b" + re.escape(k) + r"\b", t, flags=re.I)) for k in tools if re.search(r"\b" + re.escape(k) + r"\b", t, flags=re.I)})

In [ ]:
groups = {
 "software_requirements_ba": ["22","198","200","202","211","220","229","238","239","261","279"],
 "sales_or_revenue_ops": ["213","223","262","271","299"],
 "finance_or_regulatory_adjacent": ["172","181","189","270","278","282","288"],
 "engineering_or_dev_adjacent": ["117","163","245","276"],
 "data_governance_or_quality": ["126","158"],
 "mixed_or_low_quality": ["87","192","193","214","218","234","272","302"],
}
bcode = {jid: g for g, ids in groups.items() for jid in ids}
df["borderline"] = df["job_id"].isin(bcode)
df["borderline_reason"] = df["job_id"].map(bcode).fillna("")

print("borderline:", int(df["borderline"].sum()))
print(df.loc[df["borderline"], "borderline_reason"].value_counts().to_dict())
print("overlap with off_target:", df.loc[df["borderline"] & df["off_target"], "job_id"].tolist())
print("on-target rows (not off_target):", int((~df["off_target"]).sum()),
      "| of which borderline:", int((df["borderline"] & ~df["off_target"]).sum()))

In [ ]:
df.to_csv("../data_clean/stage4_flags.csv", index=False, encoding="utf-8")
chk = pd.read_csv("../data_clean/stage4_flags.csv", dtype=str, keep_default_na=False)
print(chk.shape, chk["job_id"].is_unique)
print(chk[["is_internship","is_remote","is_contract","off_target","borderline"]].apply(lambda c: (c == "True").sum()).to_dict())

In [ ]:
import re, itertools

def norm_co(s):
    s = s.lower()
    s = re.sub(r"\b(pvt|private|ltd|limited|inc|llp|llc|india|technologies|technology|solutions|services|corporation|corp)\b", " ", s)
    return re.sub(r"[^a-z0-9]", "", s)

def norm_title(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def shingles(t, k=5):
    w = re.findall(r"[a-z0-9]+", t.lower())
    return set(" ".join(w[i:i + k]) for i in range(max(1, len(w) - k + 1)))

ids = df["job_id"].tolist()
src = df["source"].tolist()
co = df["company"].tolist()
ti = df["job_title"].tolist()
ct = df["city"].tolist()
nc = [norm_co(x) for x in co]
nt = [norm_title(x) for x in ti]
sh = [shingles(t) for t in df["description_clean"]]

found = []
for a, b in itertools.combinations(range(len(df)), 2):
    inter = len(sh[a] & sh[b])
    jac = inter / len(sh[a] | sh[b]) if (sh[a] or sh[b]) else 0
    same_co = nc[a] != "" and nc[a] == nc[b]
    same_ti = nt[a] == nt[b]
    if jac >= 0.5 or (same_co and same_ti):
        found.append((round(jac, 2), ids[a], ids[b], src[a][:3] + "/" + src[b][:3], same_co, same_ti,
                      co[a][:22] + " | " + co[b][:22], ti[a][:28] + " | " + ti[b][:28], ct[a][:14] + " | " + ct[b][:14]))

found.sort(reverse=True)
print("candidate pairs:", len(found))
for f in found:
    print(*f, sep="  ")

known = [("1","4"),("30","33"),("59","72"),("64","86"),("118","159"),("145","155"),("211","238"),
         ("77","141"),("82","170"),("82","293"),("170","293"),("205","263"),("123","259"),
         ("176","303"),("166","291")]
got = {(f[1], f[2]) for f in found} | {(f[2], f[1]) for f in found}
print("known pairs NOT detected:", [k for k in known if k not in got])

In [ ]:
import re
def lines_of(t):
    return [re.sub(r"\s+", " ", l).strip() for l in t.split("\n") if len(l.strip()) > 25]

pairs = [("15","84"),("24","92"),("30","79"),("33","79"),("36","125"),("166","291"),("64","86"),("95","242")]
for a, b in pairs:
    ra = df[df["job_id"] == a].iloc[0]
    rb = df[df["job_id"] == b].iloc[0]
    la, lb = lines_of(ra["description_clean"]), lines_of(rb["description_clean"])
    sa, sb = set(la), set(lb)
    ua = [l for l in la if l not in sb]
    ub = [l for l in lb if l not in sa]
    print("\n", a, "vs", b, "| chars", len(ra["description_clean"]), len(rb["description_clean"]),
          "| lines", len(la), len(lb), "| only in first:", len(ua), "| only in second:", len(ub))
    print("   exp_portal:", ra["experience_portal"], "|", rb["experience_portal"],
          "| salary:", ra["salary_portal"][:18], "|", rb["salary_portal"][:18])
    for l in ua[:2]: print("   A:", l[:130])
    for l in ub[:2]: print("   B:", l[:130])

In [ ]:
import difflib, re

def squash(t):
    return re.sub(r"[^a-z0-9]", "", t.lower())

a = squash(df.loc[df["job_id"] == "166", "description_clean"].iloc[0])
b = squash(df.loc[df["job_id"] == "291", "description_clean"].iloc[0])
ratio = difflib.SequenceMatcher(None, a, b, autojunk=False).ratio()
print("Media.net 166 vs 291, similarity ignoring spaces and punctuation:", round(ratio, 2))

families = [["1","4"], ["30","33","79"], ["59","72"], ["64","86"], ["118","159"], ["145","155"],
            ["211","238"], ["77","141"], ["82","170","293"], ["36","125","205","263"],
            ["123","259"], ["176","303"], ["150","274"], ["14","49"], ["43","56"],
            ["122","203"], ["15","84"], ["40","198"], ["95","242"]]
probable = [["64","86"], ["77","141"], ["82","170","293"], ["176","303"], ["40","198"]]
if ratio >= 0.9:
    families.append(["166","291"])
    print("Media.net added as a duplicate pair (rule: similarity 0.9 or more)")
else:
    print("Media.net NOT added")

# 15 is the same placeholder as 84
for col, val in [("off_target", True), ("is_staffing_or_aggregator", True)]:
    df.loc[df["job_id"] == "15", col] = val
df.loc[df["job_id"] == "15", "off_target_reason"] = df.loc[df["job_id"] == "84", "off_target_reason"].iloc[0]
df.loc[df["job_id"] == "15", "staffing_reason"] = df.loc[df["job_id"] == "84", "staffing_reason"].iloc[0]

ph = {"Not mentioned", "Not disclosed", "No tags", "Confidential", ""}
cols = ["experience_portal", "salary_portal", "skills_portal", "education",
        "industry_type", "posted_ago", "work_mode_portal"]
df["fields_filled"] = sum((~df[c].isin(ph)).astype(int) for c in cols)
df["_idn"] = df["job_id"].astype(int)

df["is_repost_or_dup"] = False
df["dup_group"] = pd.NA
df["dup_kept_job_id"] = ""
df["dup_confidence"] = ""
df["dup_raw_cities"] = ""
df["dup_sources"] = ""

for n, fam in enumerate(families, 1):
    in_fam = df["job_id"].isin(fam)
    sub = df[in_fam]
    keep = sub.sort_values(["fields_filled", "_idn"], ascending=[False, True]).iloc[0]["job_id"]
    df.loc[in_fam, "dup_group"] = n
    df.loc[in_fam, "dup_kept_job_id"] = keep
    df.loc[in_fam, "dup_confidence"] = "probable" if sorted(fam) in [sorted(p) for p in probable] or fam == ["166","291"] else "exact"
    df.loc[in_fam & (df["job_id"] != keep), "is_repost_or_dup"] = True
    df.loc[df["job_id"] == keep, "dup_raw_cities"] = " | ".join(sub["city"])
    df.loc[df["job_id"] == keep, "dup_sources"] = " + ".join(sorted(set(sub["source"])))

print("groups:", len(families), "| rows flagged is_repost_or_dup:", int(df["is_repost_or_dup"].sum()))
print("by source:", df.loc[df["is_repost_or_dup"], "source"].value_counts().to_dict())
print("kept rows of probable groups:", df.loc[(df["dup_confidence"] == "probable") & ~df["is_repost_or_dup"], "job_id"].tolist())
print("off_target now:", int(df["off_target"].sum()), "| staffing now:", int(df["is_staffing_or_aggregator"].sum()))
print("unique postings (not dup):", int((~df["is_repost_or_dup"]).sum()))
print("analysis set (not off_target and not dup):", int((~df["off_target"] & ~df["is_repost_or_dup"]).sum()))
print("analysis set by source:", df.loc[~df["off_target"] & ~df["is_repost_or_dup"], "source"].value_counts().to_dict())

In [ ]:
df.loc[df["dup_group"].notna() & df["job_id"].isin(["166", "291"]), "dup_confidence"] = "exact"
df.drop(columns=["_idn"]).to_csv("../data_clean/stage5_dedup.csv", index=False, encoding="utf-8")
chk = pd.read_csv("../data_clean/stage5_dedup.csv", dtype=str, keep_default_na=False)
print(chk.shape, chk["job_id"].is_unique)
print((chk["is_repost_or_dup"] == "True").sum(), (chk["off_target"] == "True").sum())

In [ ]:
blocks = [
 (1, 10, "pilot"), (11, 30, "Bengaluru"), (31, 44, "Hyderabad"), (45, 61, "Pune"),
 (62, 76, "Chennai"), (77, 90, "Mumbai"), (91, 104, "Delhi/NCR"), (105, 108, "Bengaluru"),
 (109, 119, "Bengaluru"), (120, 131, "Hyderabad"), (132, 146, "Pune"), (147, 159, "Chennai"),
 (160, 171, "Mumbai"), (172, 184, "Delhi/NCR"),
 (185, 196, "Bengaluru"), (197, 206, "Hyderabad"), (207, 216, "Pune"), (217, 225, "Chennai"),
 (226, 233, "Mumbai"), (234, 244, "Delhi/NCR"),
 (245, 255, "Bengaluru"), (256, 264, "Hyderabad"), (265, 273, "Pune"), (274, 283, "Chennai"),
 (284, 293, "Mumbai"), (294, 303, "Delhi/NCR"),
]
def block_city(j):
    j = int(j)
    for lo, hi, c in blocks:
        if lo <= j <= hi:
            return c
    return "unknown"

df["search_city"] = df["job_id"].map(block_city)
print(df["search_city"].value_counts().to_dict())
print("pilot rows (jobs 1-10) by source:", df.loc[df["search_city"] == "pilot", "source"].value_counts().to_dict())

print("--- 40 most common raw city strings")
print(df["city"].value_counts().head(40).to_string())

print("--- strings that contain none of the six metro names")
metro = r"bengaluru|bangalore|hyderabad|pune|pimpri|chennai|mumbai|thane|delhi|gurugram|gurgaon|noida|ghaziabad|faridabad"
odd = df[~df["city"].str.contains(metro, case=False, regex=True)]
print(len(odd), "rows")
for jid, src, c, s in zip(odd["job_id"], odd["source"], odd["city"], odd["search_city"]):
    print(jid, src, "|", c[:60], "| search:", s)

In [ ]:
import re

metros = {
 "Bengaluru": r"bengaluru|bangalore",
 "Hyderabad": r"hyderabad|secunderabad|serilingampalli",
 "Pune": r"\bpune\b|pimpri|chinchwad",
 "Chennai": r"chennai",
 "Mumbai": r"mumbai|\bthane\b",
 "Delhi/NCR": r"delhi|gurugram|gurgaon|noida|ghaziabad|faridabad",
}
def metros_in(s):
    return [m for m, p in metros.items() if re.search(p, s, flags=re.I)]

df["city_metros"] = df["city"].map(lambda s: "|".join(metros_in(s)))

primary, rule = [], []
for jid, sc, cm in zip(df["job_id"], df["search_city"], df["city_metros"]):
    found = cm.split("|") if cm else []
    if len(found) == 1:
        if sc in ("pilot", found[0]):
            primary.append(found[0]); rule.append("1_single_metro")
        else:
            primary.append(""); rule.append("4_conflict")
    elif len(found) > 1:
        if sc in found:
            primary.append(sc); rule.append("2_multi_city")
        else:
            primary.append(""); rule.append("4_conflict" if sc != "pilot" else "5_pilot_manual")
    else:
        if sc == "pilot":
            primary.append(""); rule.append("5_pilot_manual")
        else:
            primary.append(sc); rule.append("3_no_metro_in_field")

df["primary_city"] = primary
df["city_rule"] = rule
print(df["city_rule"].value_counts().to_dict())
print("primary_city filled:", int((df["primary_city"] != "").sum()))
print(df.loc[df["primary_city"] != "", "primary_city"].value_counts().to_dict())

print("--- rows needing a decision (rules 4 and 5)")
need = df[df["city_rule"].isin(["4_conflict", "5_pilot_manual"])]
for jid, src, c, s, cm, dup, off in zip(need["job_id"], need["source"], need["city"], need["search_city"],
                                        need["city_metros"], need["is_repost_or_dup"], need["off_target"]):
    print(jid, src, "| field:", c[:45], "| metros in field:", cm or "-", "| search:", s, "| dup:", dup, "| off_target:", off)

In [ ]:
manual = {
 "1": ("Bengaluru", "city_assumed"), "4": ("Bengaluru", "city_assumed"),
 "10": ("Other", "outside_six_metros"),
 "108": ("Hyderabad", "city_differs_from_search"),
 "141": ("Mumbai", "city_differs_from_search"),
 "144": ("Mumbai", "city_differs_from_search"),
 "146": ("Mumbai", "city_differs_from_search"),
}
df["city_flag"] = ""
for jid, (c, f) in manual.items():
    df.loc[df["job_id"] == jid, "primary_city"] = c
    df.loc[df["job_id"] == jid, "city_flag"] = f
df.loc[df["city_rule"] == "2_multi_city", "city_flag"] = "multi_city_posting"
df.loc[df["city_rule"] == "3_no_metro_in_field", "city_flag"] = "city_field_no_metro"
print("primary_city blank (should be only job 8):", df.loc[df["primary_city"] == "", "job_id"].tolist())
print(df["city_flag"].value_counts().to_dict())

others = r"ahmedabad|kolkata|nagpur|jaipur|kochi|cochin|coimbatore|indore|lucknow|chandigarh|bhubaneswar|vadodara|surat|nashik|mysuru|mysore|visakhapatnam|trivandrum|thiruvananthapuram"
print("--- location mentions in descriptions (first 2 each)")
for jid in ["8", "37", "43", "56"]:
    r = df[df["job_id"] == jid].iloc[0]
    print("\n", jid, "| title:", r["job_title"][:50], "| field:", r["city"][:50], "| primary:", r["primary_city"])
    t = r["description_clean"]
    for m in list(re.finditer(others + "|" + "|".join(metros.values()) + r"|location", t, flags=re.I))[:2]:
        s = max(0, m.start() - 50)
        print("   ", repr(t[s:m.end() + 50]))

print("\n--- titles naming a non-metro city")
mt = df["job_title"].str.contains(others, case=False, regex=True)
for jid, t, c, p in zip(df.loc[mt, "job_id"], df.loc[mt, "job_title"], df.loc[mt, "city"], df.loc[mt, "primary_city"]):
    print(jid, "|", t[:60], "| field:", c[:30], "| primary:", p)

In [ ]:
for jid, c, f in [("8", "Unknown", "city_unknown"),
                  ("43", "Other", "city_field_conflicts_description"),
                  ("56", "Other", "city_field_conflicts_description")]:
    df.loc[df["job_id"] == jid, "primary_city"] = c
    df.loc[df["job_id"] == jid, "city_flag"] = f

print("all 300 rows:", df["primary_city"].value_counts().to_dict())
ana = ~df["off_target"] & ~df["is_repost_or_dup"]
print("analysis set (", int(ana.sum()), "):", df.loc[ana, "primary_city"].value_counts().to_dict())
print("analysis set by city and source:")
print(pd.crosstab(df.loc[ana, "primary_city"], df.loc[ana, "source"]))
print("blank primary_city:", int((df["primary_city"] == "").sum()))

df.drop(columns=["_idn"], errors="ignore").to_csv("../data_clean/stage7_city.csv", index=False, encoding="utf-8")
chk = pd.read_csv("../data_clean/stage7_city.csv", dtype=str, keep_default_na=False)
print(chk.shape, chk["job_id"].is_unique)

In [ ]:
print("--- date_collected by source")
print(df.groupby("source")["date_collected"].value_counts().sort_index().to_string())

print("--- posted_ago, all distinct values")
print(df["posted_ago"].value_counts().to_string())

def nk_id(u):
    seg = u.split("?")[0].rstrip("/").split("-")[-1]
    return seg if re.fullmatch(r"\d{12}", seg) else ""

nk = df["source"] == "Naukri"
df["nk_url_id"] = df["url"].map(nk_id).where(nk, "")
df["nk_url_date"] = pd.to_datetime(df["nk_url_id"].str[:6], format="%d%m%y", errors="coerce")
print("--- Naukri URL IDs: parsed", int(df.loc[nk, "nk_url_date"].notna().sum()), "of", int(nk.sum()))
print("unparsed job_ids:", df.loc[nk & df["nk_url_date"].isna(), "job_id"].tolist())
print("range:", df.loc[nk, "nk_url_date"].min(), "to", df.loc[nk, "nk_url_date"].max())
print(df.loc[nk, "nk_url_date"].dt.to_period("M").value_counts().sort_index().to_string())
print("--- sample: URL date vs posted_ago")
print(df.loc[nk & df["nk_url_date"].notna(), ["job_id", "nk_url_id", "nk_url_date", "posted_ago"]].sample(12, random_state=1).to_string())

In [ ]:
df["collected"] = pd.to_datetime(df["date_collected"], errors="coerce")
print("unparsed date_collected:", int(df["collected"].isna().sum()))

def label_days(s):
    t = s.lower().strip()
    t = re.sub(r"^(posted|reposted)\s+", "", t)
    if t in ("not mentioned", ""):
        return None
    if t in ("today", "just now", "few hours ago"):
        return 0
    if t == "a month ago":
        return 30
    m = re.fullmatch(r"(\d+)\+?\s*(minute|hour|day|week|month)s?\s+ago", t)
    if m:
        n = int(m.group(1))
        return n * {"minute": 0, "hour": 0, "day": 1, "week": 7, "month": 30}[m.group(2)]
    return None

df["age_label_days"] = df["posted_ago"].map(label_days)
df["label_open_ended"] = df["posted_ago"].str.contains(r"\+", regex=True)
df["is_reposted_label"] = df["posted_ago"].str.startswith("Reposted")
bad = df[(df["posted_ago"] != "Not mentioned") & df["age_label_days"].isna()]
print("posted_ago values the parser could not read:", bad["posted_ago"].unique().tolist())

df["nk_url_age"] = (df["collected"] - df["nk_url_date"]).dt.days
use_url = (df["source"] == "Naukri") & df["label_open_ended"]
df["age_days"] = df["age_label_days"].where(~use_url, df["nk_url_age"])
df["age_source"] = "none"
df.loc[df["age_label_days"].notna(), "age_source"] = "label"
df.loc[use_url, "age_source"] = "naukri_url"
print("age_source:", df["age_source"].value_counts().to_dict())
print("age known:", int(df["age_days"].notna().sum()), "| by source:", df.loc[df["age_days"].notna(), "source"].value_counts().to_dict())
print("reposted labels:", int(df["is_reposted_label"].sum()), df.loc[df["is_reposted_label"], "job_id"].tolist())

print("--- Naukri rows where the label is specific but the URL date is much older (refreshed)")
ref = (df["source"] == "Naukri") & ~df["label_open_ended"] & ((df["nk_url_age"] - df["age_label_days"]) > 30)
print(int(ref.sum()), "rows")
for jid, lab, d in zip(df.loc[ref, "job_id"], df.loc[ref, "posted_ago"], df.loc[ref, "nk_url_date"]):
    print(jid, "|", lab, "| URL date:", d.date())

print("--- rows at or above each cutoff (age known only)")
known = df["age_days"].notna()
for c in [30, 45, 60, 90]:
    o = known & (df["age_days"] >= c)
    print(c, "days:", int(o.sum()), "of", int(known.sum()), "| by source:", df.loc[o, "source"].value_counts().to_dict())

In [ ]:
df["age_known"] = df["age_days"].notna()
df["is_old"] = (df["age_days"] >= 60).where(df["age_known"])
df["refreshed_label_conflict"] = ref

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
print("is_old (>= 60 days), all rows:", int((df["is_old"] == True).sum()), "of", int(df["age_known"].sum()), "with a known age")
print("analysis set:", int(ana.sum()), "| age known:", int((ana & df["age_known"]).sum()),
      "| old:", int((ana & (df["is_old"] == True)).sum()))
print("analysis set, old by source:", df.loc[ana & (df["is_old"] == True), "source"].value_counts().to_dict())
print("analysis set, age unknown (Indeed):", int((ana & ~df["age_known"]).sum()))
print("analysis set, reposted label:", int((ana & df["is_reposted_label"]).sum()))
print("age_days among known rows:", df.loc[df["age_known"], "age_days"].describe()[["min", "50%", "max"]].round().to_dict())

df.to_csv("../data_clean/stage8_dates.csv", index=False, encoding="utf-8")
chk = pd.read_csv("../data_clean/stage8_dates.csv", dtype=str, keep_default_na=False)
print(chk.shape, chk["job_id"].is_unique)

In [ ]:
pat = re.compile(r"[^.]{0,25}\b\d+(?:\.\d+)?\s*(?:[-–—]|to)?\s*(?:\d+(?:\.\d+)?)?\s*\+?\s*(?:years?|yrs?|months?)\b[^.]{0,25}", re.I)

from collections import Counter
shape_counts = Counter()
rows_with = set()
for jid, t in zip(df["job_id"], df["description_clean"]):
    for m in pat.finditer(t):
        s = re.sub(r"\d+(?:\.\d+)?", "N", m.group(0))
        s = re.sub(r"\s+", " ", s).strip().lower()
        shape_counts[s] += 1
        rows_with.add(jid)

print("rows with at least one number-plus-years/months phrase:", len(rows_with), "of 300")
print("distinct phrase shapes:", len(shape_counts))
for s, n in shape_counts.most_common(60):
    print(n, "|", s[:75])

none = df[~df["job_id"].isin(rows_with)]
print("rows with no such phrase:", len(none), "| by source:", none["source"].value_counts().to_dict())
word = none["description_clean"].str.contains(r"fresher|freshers|entry[- ]level|no experience|recent graduate|new graduate|students", case=False, regex=True)
print("   of those, mention fresher/entry-level/graduate/students:", int(word.sum()))
print("   job_ids with no phrase and no fresher word:", none.loc[~word, "job_id"].tolist()[:50])

In [ ]:
none = df[~df["job_id"].isin(rows_with)].copy()
exp_word = none["description_clean"].str.contains(r"experience|experienced|\bexp\b", case=False, regex=True)
fresh_word = none["description_clean"].str.contains(r"fresher|freshers|entry[- ]level|no experience|recent graduate|new graduate|students", case=False, regex=True)
print("no-phrase rows:", len(none))
print("  mention experience/exp:", int(exp_word.sum()), "| fresher-type wording:", int(fresh_word.sum()),
      "| neither:", int((~exp_word & ~fresh_word).sum()))
print("  neither, by source:", none.loc[~exp_word & ~fresh_word, "source"].value_counts().to_dict())

nums = r"(?:one|two|three|four|five|six|seven|eight|nine|ten|twelve)"
sp = re.compile(r"\b" + nums + r"(?:\s*(?:to|or|-|–)\s*" + nums + r")?\s*\+?\s*(?:years?|yrs?|months?)\b", re.I)
plus = re.compile(r"\b\d+\s*plus\s*(?:years?|yrs?)\b", re.I)
print("--- spelled-out numbers and N plus years, all rows")
for jid, t in zip(df["job_id"], df["description_clean"]):
    for rx in (sp, plus):
        m = rx.search(t)
        if m:
            s = max(0, m.start() - 35)
            print(jid, repr(t[s:m.end() + 35]))

print("--- sample of no-phrase rows that mention experience (snippet around the word)")
samp = none[exp_word].sample(min(15, int(exp_word.sum())), random_state=2)
for jid, src, t in zip(samp["job_id"], samp["source"], samp["description_clean"]):
    m = re.search(r"experience|experienced|\bexp\b", t, flags=re.I)
    s = max(0, m.start() - 60)
    print(jid, src, repr(t[s:m.end() + 60]))

In [ ]:
import re, numpy as np

NUM = r"\d+(?:\.\d+)?"
CAND = re.compile(r"(?<![\d.])(?P<a>" + NUM + r")\s*(?:-|to)?\s*(?P<b>" + NUM + r")?\s*(?P<plus>\+)?\s*"
                  r"(?P<unit>years?|yrs?|months?|mths?)(?-i:(?![a-z]))", re.I)
INCL = re.compile(r"experience|\bexp\b", re.I)
EXCL = re.compile(r"agreement|commitment|\bbond\b|horizon|scaled|signed up|in a row|past (?:\w+ ){0,2}years|"
                  r"duration|probation|notice period|lock[- ]in|stipend", re.I)
UPTO = re.compile(r"(?:up to|upto|less than|\bunder\b|maximum|\bmax\b|within|below|not more than|<)$", re.I)
MINQ = re.compile(r"(?:minimum(?: of)?|at least|atleast|\bmin\b\.?|over|more than)$", re.I)

def candidates(t):
    t2 = t.replace("\u2013", "-").replace("\u2014", "-").replace("\u2011", "-")
    out, excl = [], []
    for m in CAND.finditer(t2):
        s, e = m.start(), m.end()
        w = t2[max(0, s - 50): e + 40]
        a = float(m.group("a"))
        unit = m.group("unit").lower()
        div = 12.0 if unit.startswith("m") else 1.0
        if EXCL.search(w):
            excl.append(re.sub(r"\s+", " ", t2[max(0, s - 25): e + 15]))
            continue
        if not INCL.search(w):
            continue
        pre = t2[max(0, s - 30): s].rstrip()
        if m.group("b") is not None:
            bb = float(m.group("b"))
            if bb <= a:
                continue
            lo, hi, form = a / div, bb / div, "range"
        elif UPTO.search(pre):
            lo, hi, form = 0.0, a / div, "upto"
        elif m.group("plus"):
            lo, hi, form = a / div, None, "plus"
        elif MINQ.search(pre):
            lo, hi, form = a / div, None, "min"
        else:
            lo, hi, form = a / div, None, "single"
        alt = bool(re.search(r"bachelor|master|\bphd\b|\bmba\b|b\.?tech", w, re.I))
        out.append((lo, hi, form, alt, re.sub(r"\s+", " ", t2[max(0, s - 30): e + 10])))
    return out, excl

res = df["description_clean"].map(candidates)
df["_cands"] = res.map(lambda x: x[0])
df["_excl"] = res.map(lambda x: x[1])
first = df["_cands"].map(lambda c: c[0] if c else None)
df["exp_min"] = first.map(lambda c: c[0] if c else np.nan)
df["exp_max"] = first.map(lambda c: c[1] if c and c[1] is not None else np.nan)
df["exp_form"] = first.map(lambda c: c[2] if c else "")
df["exp_n_phrases"] = df["_cands"].map(len)
df["exp_phrases_disagree"] = df["_cands"].map(lambda c: len({x[0] for x in c}) > 1)
df["exp_alt_degree"] = df["_cands"].map(lambda c: len(c) > 1 and any(x[3] for x in c))
df["exp_source"] = np.where(df["exp_min"].notna(), "description", "")

lab = df["description_clean"].str.contains(r"experience\s*:?\s*(?:-\s*)?fresher", case=False, regex=True)
need = df["exp_min"].isna() & lab
df.loc[need, "exp_min"] = 0.0
df.loc[need, "exp_form"] = "fresher_label"
df.loc[need, "exp_source"] = "fresher_label"

manual = {"169": (1.0, np.nan, "min"), "197": (1.0, np.nan, "min"),
          "100": (1.0, 2.0, "range"), "61": (2.0, 4.0, "range")}
for jid, (lo, hi, f) in manual.items():
    i = df["job_id"] == jid
    df.loc[i, "exp_min"] = lo
    df.loc[i, "exp_max"] = hi
    df.loc[i, "exp_form"] = f
    df.loc[i, "exp_source"] = "manual"

entry = df["description_clean"].str.contains(
    r"\bfreshers?\b|entry[- ]level|no prior experience|recent graduates?|new graduates?|fresh graduates?|fresh grads?",
    case=False, regex=True)
df["exp_entry_wording"] = df["exp_min"].isna() & entry

print("exp_min filled:", int(df["exp_min"].notna().sum()), "of 300 | by source:",
      df.loc[df["exp_min"].notna(), "source"].value_counts().to_dict())
print("exp_source:", df["exp_source"].value_counts().to_dict())
print("exp_form:", df["exp_form"].value_counts().to_dict())
print("null exp_min:", int(df["exp_min"].isna().sum()), "| of those entry wording only:", int(df["exp_entry_wording"].sum()))

print("--- exp_min of 10 or more (possible lost-dash errors)")
for jid, lo, f in zip(df.loc[df["exp_min"] >= 10, "job_id"], df.loc[df["exp_min"] >= 10, "exp_min"], df.loc[df["exp_min"] >= 10, "exp_form"]):
    print(jid, lo, f)

print("--- rows whose phrases disagree on the minimum:", int(df["exp_phrases_disagree"].sum()))
for jid, c, alt in zip(df.loc[df["exp_phrases_disagree"], "job_id"], df.loc[df["exp_phrases_disagree"], "_cands"],
                       df.loc[df["exp_phrases_disagree"], "exp_alt_degree"]):
    print(jid, [(round(x[0], 2), None if x[1] is None else round(x[1], 2), x[2]) for x in c], "| degree alternatives:", alt)

print("--- phrases rejected by the exclusion rule:", int(df["_excl"].map(len).sum()))
k = 0
for jid, ex in zip(df["job_id"], df["_excl"]):
    for s in ex:
        if k < 25:
            print(jid, repr(s))
        k += 1

print("--- random sample of 15 parsed rows")
sm = df[df["exp_source"] == "description"].sample(15, random_state=3)
for jid, lo, hi, f, c in zip(sm["job_id"], sm["exp_min"], sm["exp_max"], sm["exp_form"], sm["_cands"]):
    print(jid, "->", lo, hi, f, "|", repr(c[0][4]))

In [ ]:
i = df["job_id"] == "208"
df.loc[i, ["exp_min", "exp_max"]] = [9.0, 12.0]
df.loc[i, "exp_form"] = "range"
df.loc[i, "exp_source"] = "manual"

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
print("exp_min filled:", int(df["exp_min"].notna().sum()), "of 300 | analysis set:",
      int((ana & df["exp_min"].notna()).sum()), "of", int(ana.sum()))

print("--- phrases of the unresolved disagreeing rows")
for jid in ["7", "58", "78", "80", "231", "243", "245", "267"]:
    c = df.loc[df["job_id"] == jid, "_cands"].iloc[0]
    print(jid)
    for x in c:
        print("   ", round(x[0], 2), None if x[1] is None else round(x[1], 2), x[2], "|", repr(x[4]))

print("--- wide context for 2 and 21")
for jid, a in [("2", 14.0), ("21", 12.0)]:
    t2 = df.loc[df["job_id"] == jid, "description_clean"].iloc[0].replace("\u2013", "-")
    for m in CAND.finditer(t2):
        if float(m.group("a")) == a:
            print(jid, repr(t2[max(0, m.start() - 110): m.end() + 70]))
            break

print("--- rows with a years/months phrase but no exp_min")
miss = df[df["job_id"].isin(rows_with) & df["exp_min"].isna()]
print(len(miss), "rows")
for jid, t in zip(miss["job_id"], miss["description_clean"]):
    hits = [re.sub(r"\s+", " ", m.group(0))[:60] for m in list(pat.finditer(t))[:2]]
    print(jid, hits)

print("--- Fresher (N years) labels")
for jid, t, lo in zip(df["job_id"], df["description_clean"], df["exp_min"]):
    m = re.search(r"fresher\s*\(\s*\d[^)]*\)", t, flags=re.I)
    if m:
        print(jid, "exp_min parsed:", lo, "|", repr(t[max(0, m.start() - 30): m.end() + 20]))

In [ ]:
manual2 = {
 "7": (1.0, 2.0, "range"), "58": (0.5, 1.0, "range"), "78": (0.5, 1.0, "range"),
 "231": (8.0, 10.0, "range"), "245": (4.0, np.nan, "min"),
 "29": (4.0, np.nan, "plus"), "140": (6.0, np.nan, "plus"), "250": (1.0, np.nan, "plus"),
 "261": (2.0, 3.0, "range"), "275": (6.0, np.nan, "plus"),
}
for jid, (lo, hi, f) in manual2.items():
    i = df["job_id"] == jid
    df.loc[i, ["exp_min", "exp_max"]] = [lo, hi]
    df.loc[i, "exp_form"] = f
    df.loc[i, "exp_source"] = "manual"

def lowest(c):
    x = min(c, key=lambda z: z[0])
    return x[0], (np.nan if x[1] is None else x[1]), x[2]

rule_rows = df.loc[df["exp_phrases_disagree"] & (df["exp_source"] == "description") & (df["job_id"] != "2"), "job_id"].tolist()
print("lowest-minimum rule applied to:", rule_rows)
for jid in rule_rows:
    i = df["job_id"] == jid
    lo, hi, f = lowest(df.loc[i, "_cands"].iloc[0])
    df.loc[i, ["exp_min", "exp_max"]] = [lo, hi]
    df.loc[i, "exp_form"] = f
    print("  ", jid, "->", lo, hi, f)

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
print("exp_min filled:", int(df["exp_min"].notna().sum()), "of 300 | analysis set:",
      int((ana & df["exp_min"].notna()).sum()), "of", int(ana.sum()))
print("exp_source:", df["exp_source"].value_counts().to_dict())
print("analysis set filled by source:", df.loc[ana & df["exp_min"].notna(), "source"].value_counts().to_dict())
print("analysis set null by source:", df.loc[ana & df["exp_min"].isna(), "source"].value_counts().to_dict())
print("job 2 portal experience:", df.loc[df["job_id"] == "2", "experience_portal"].iloc[0],
      "| job 21:", df.loc[df["job_id"] == "21", "experience_portal"].iloc[0],
      "| job 153:", df.loc[df["job_id"] == "153", "experience_portal"].iloc[0])

In [ ]:
i = df["job_id"] == "2"
df.loc[i, ["exp_min", "exp_max"]] = [np.nan, np.nan]
df.loc[i, "exp_form"] = ""
df.loc[i, "exp_source"] = "lost_dash_null"
df["exp_lost_dash"] = df["job_id"].isin(["2", "100", "58", "61"])

def portal_exp(s):
    t = s.lower().replace("\u2013", "-")
    m = re.search(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)", t)
    if m:
        return float(m.group(1)), float(m.group(2))
    if "fresher" in t:
        return 0.0, np.nan
    m = re.search(r"(\d+(?:\.\d+)?)\s*years?", t)
    if m:
        return float(m.group(1)), np.nan
    return np.nan, np.nan

pe = df["experience_portal"].map(portal_exp)
df["exp_portal_min"] = pe.map(lambda x: x[0])
df["exp_portal_max"] = pe.map(lambda x: x[1])

tab = pd.DataFrame({"value": df["experience_portal"], "min": df["exp_portal_min"], "max": df["exp_portal_max"]})
print(tab.drop_duplicates().sort_values("value").to_string(index=False))

df["exp_min_fallback"] = df["exp_min"].fillna(df["exp_portal_min"])
df["exp_fallback_source"] = np.where(df["exp_min"].notna(), "description",
                             np.where(df["exp_portal_min"].notna(), "portal", "none"))

both = df["exp_min"].notna() & df["exp_portal_min"].notna()
df["exp_conflict_flag"] = (both & ((df["exp_min"] - df["exp_portal_min"]).abs() >= 1)).where(both)
df["desc_min_above_portal_max"] = (both & df["exp_portal_max"].notna() & (df["exp_min"] > df["exp_portal_max"])).where(both)

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
print("exp_min filled (all):", int(df["exp_min"].notna().sum()))
print("analysis set (246): description", int((ana & (df["exp_fallback_source"] == "description")).sum()),
      "| portal fallback", int((ana & (df["exp_fallback_source"] == "portal")).sum()),
      "| none", int((ana & (df["exp_fallback_source"] == "none")).sum()))
print("analysis set both values present:", int((ana & both).sum()),
      "| differ by 1+ year:", int((ana & (df["exp_conflict_flag"] == True)).sum()),
      "| description min above portal max:", int((ana & (df["desc_min_above_portal_max"] == True)).sum()))
print("by source (both present, differ):")
print(df[ana & both].groupby("source")["exp_conflict_flag"].agg(["sum", "count"]).to_string())

df.drop(columns=["_cands", "_excl"]).to_csv("../data_clean/stage9_experience.csv", index=False, encoding="utf-8")
chk = pd.read_csv("../data_clean/stage9_experience.csv", dtype=str, keep_default_na=False)
print(chk.shape, chk["job_id"].is_unique)

In [ ]:
LEVEL_WORDS = re.compile(r"\b(fresher|freshers|junior|jr|trainee|trainees|intern|interns|internship)\b", re.I)
ASSOC = re.compile(r"\bassociate\b", re.I)
LEVEL_I = re.compile(r"(?<![A-Za-z0-9])(?:I|1)\s*$|(?:analyst|executive|associate|engineer)\s*[-]?\s*(?:I|1)(?![A-Za-z0-9])", re.I)

tt = df["job_title"].str.replace("\u2013", "-", regex=False)
df["entry_title_words"] = tt.map(lambda t: bool(LEVEL_WORDS.search(t)))
df["entry_title_assoc"] = tt.map(lambda t: bool(ASSOC.search(t)))
df["entry_title_level"] = tt.map(lambda t: bool(LEVEL_I.search(t)))
df["entry_title_any"] = df["entry_title_words"] | df["entry_title_assoc"] | df["entry_title_level"]
df["entry_title_no_assoc"] = df["entry_title_words"] | df["entry_title_level"]

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
print("all rows: any", int(df["entry_title_any"].sum()), "| without Associate", int(df["entry_title_no_assoc"].sum()),
      "| Associate only", int((df["entry_title_assoc"] & ~df["entry_title_no_assoc"]).sum()))
print("analysis set (246): any", int((ana & df["entry_title_any"]).sum()),
      "| without Associate", int((ana & df["entry_title_no_assoc"]).sum()),
      "| Associate only", int((ana & df["entry_title_assoc"] & ~df["entry_title_no_assoc"]).sum()))
print("analysis set entry-level by source:", df.loc[ana & df["entry_title_any"], "source"].value_counts().to_dict())
print("analysis set entry-level by city:", df.loc[ana & df["entry_title_any"], "primary_city"].value_counts().to_dict())

print("--- entry-level titles in the analysis set")
sub = df[ana & df["entry_title_any"]].sort_values("job_id", key=lambda s: s.astype(int))
for jid, src, t, w, a, l in zip(sub["job_id"], sub["source"], sub["job_title"], sub["entry_title_words"], sub["entry_title_assoc"], sub["entry_title_level"]):
    print(jid, src[:3], "W" if w else "-", "A" if a else "-", "L" if l else "-", "|", t[:62])

print("--- data for Q4 on the entry-level rows (analysis set)")
e = ana & df["entry_title_any"]
print("rows:", int(e.sum()), "| description exp_min known:", int((e & df["exp_min"].notna()).sum()),
      "| portal minimum known:", int((e & df["exp_portal_min"].notna()).sum()),
      "| both:", int((e & df["exp_min"].notna() & df["exp_portal_min"].notna()).sum()),
      "| neither:", int((e & df["exp_min"].isna() & df["exp_portal_min"].isna()).sum()))

print("--- titles hinting at entry level that did not match the rule (analysis set)")
hint = df["job_title"].str.contains(r"entry|graduate|batch|20\d\d|\d\s*-\s*\d\s*year|\d\+?\s*year|trainee|apprentice", case=False, regex=True)
for jid, src, t in zip(df.loc[ana & hint & ~df["entry_title_any"], "job_id"], df.loc[ana & hint & ~df["entry_title_any"], "source"], df.loc[ana & hint & ~df["entry_title_any"], "job_title"]):
    print(jid, src[:3], "|", t[:70])

print("--- description min above portal max, analysis set:", df.loc[ana & (df["desc_min_above_portal_max"] == True), "job_id"].tolist())

In [ ]:
tt = df["job_title"].str.replace("\u2013", "-", regex=False).str.replace("_", " ", regex=False)
df["entry_title_words"] = tt.map(lambda t: bool(LEVEL_WORDS.search(t)))
df["entry_title_assoc"] = tt.map(lambda t: bool(ASSOC.search(t)))
df["entry_title_level"] = tt.map(lambda t: bool(LEVEL_I.search(t)))
df["entry_title_any"] = df["entry_title_words"] | df["entry_title_assoc"] | df["entry_title_level"]
df["entry_title_no_assoc"] = df["entry_title_words"] | df["entry_title_level"]
df["entry_title_mixed"] = df["entry_title_words"] & tt.str.contains(r"experienced|\bexp\b", case=False, regex=True)

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
print("analysis set: any", int((ana & df["entry_title_any"]).sum()),
      "| without Associate", int((ana & df["entry_title_no_assoc"]).sum()),
      "| Associate only", int((ana & df["entry_title_assoc"] & ~df["entry_title_no_assoc"]).sum()))
print("mixed-title rows:", df.loc[ana & df["entry_title_mixed"], "job_id"].tolist())

for jid in ["38", "39"]:
    r = df[df["job_id"] == jid].iloc[0]
    print(jid, "| title:", r["job_title"], "| entry_title_any:", r["entry_title_any"],
          "| in analysis set:", bool(ana[df["job_id"] == jid].iloc[0]),
          "| off_target:", r["off_target"], "| dup:", r["is_repost_or_dup"], "| internship:", r["is_internship"])

groups = {
 "any title rule": ana & df["entry_title_any"],
 "without Associate": ana & df["entry_title_no_assoc"],
 "without Associate, without internships": ana & df["entry_title_no_assoc"] & ~df["is_internship"],
}
for name, g in groups.items():
    print(name, "| rows", int(g.sum()),
          "| description exp known", int((g & df["exp_min"].notna()).sum()),
          "| portal min known", int((g & df["exp_portal_min"].notna()).sum()),
          "| both", int((g & df["exp_min"].notna() & df["exp_portal_min"].notna()).sum()),
          "| neither", int((g & df["exp_min"].isna() & df["exp_portal_min"].isna()).sum()))
print("without Associate, by source:", df.loc[groups["without Associate"], "source"].value_counts().to_dict())
print("internship-flagged rows inside the entry-level set:", int((groups["any title rule"] & df["is_internship"]).sum()))

df.drop(columns=["_cands", "_excl"], errors="ignore").to_csv("../data_clean/stage10_entry.csv", index=False, encoding="utf-8")
chk = pd.read_csv("../data_clean/stage10_entry.csv", dtype=str, keep_default_na=False)
print(chk.shape, chk["job_id"].is_unique)

In [ ]:
ana = ~df["off_target"] & ~df["is_repost_or_dup"]
has = df["salary_portal"] != "Not disclosed"
print("portal salary present:", int(has.sum()), "of 300 | analysis set:", int((ana & has).sum()), "of", int(ana.sum()))
print("by source (all rows):", df.loc[has, "source"].value_counts().to_dict())

print("--- distinct portal salary strings")
tab = df[has].groupby(["source", "salary_portal"]).size().reset_index(name="n")
for s, v, n in zip(tab["source"], tab["salary_portal"], tab["n"]):
    print(s[:3], n, "|", v)

print("--- salary figures inside descriptions where the portal says Not disclosed")
cur = re.compile(r"(?:₹|\brs\.?|\binr\b)\s*\d[\d,\.]*", re.I)
unit = re.compile(r"\d[\d,\.]*\s*(?:k\b|lpa\b|lacs?\b|lakhs?\b)|\bctc\b|\blpa\b", re.I)
k = 0
for jid, src, t in zip(df.loc[~has, "job_id"], df.loc[~has, "source"], df.loc[~has, "description_text"]):
    m = cur.search(t) or unit.search(t)
    if m:
        k += 1
        s = max(0, m.start() - 40)
        print(jid, src[:3], repr(re.sub(r"\s+", " ", t[s: m.end() + 40])))
print("rows with a salary-like figure in the description but no portal salary:", k)

In [ ]:
def lakh_num(x):
    v = float(x.replace(",", ""))
    return v / 100000.0 if ("," in x and v >= 1000) else v

def per(period):
    return 12 if period.lower() == "month" else 1

def parse_salary(raw):
    s = raw.replace("\u20b9", "").strip()
    if s in ("Not disclosed", "", "Not mentioned"):
        return None
    m = re.fullmatch(r"([\d,\.]+)\s*-\s*([\d,\.]+)\s*(?:Lacs P\.A\.|LPA)", s, flags=re.I)
    if m:
        return lakh_num(m.group(1)), lakh_num(m.group(2)), "lakh_range"
    m = re.fullmatch(r"([\d,\.]+)\s*(?:Lacs P\.A\.|LPA)", s, flags=re.I)
    if m:
        v = lakh_num(m.group(1)); return v, v, "lakh_single"
    m = re.fullmatch(r"From\s*([\d,]+)\s*a\s*(year|month)", s, flags=re.I)
    if m:
        return float(m.group(1).replace(",", "")) * per(m.group(2)) / 1e5, None, "from_min_only"
    m = re.fullmatch(r"Up to\s*([\d,]+)\s*a\s*(year|month)", s, flags=re.I)
    if m:
        return None, float(m.group(1).replace(",", "")) * per(m.group(2)) / 1e5, "upto_max_only"
    m = re.fullmatch(r"([\d,]+)\s*-\s*([\d,]+)\s*a\s*(year|month)", s, flags=re.I)
    if m:
        f = per(m.group(3))
        return (float(m.group(1).replace(",", "")) * f / 1e5, float(m.group(2).replace(",", "")) * f / 1e5,
                "monthly_range" if f == 12 else "annual_range")
    m = re.fullmatch(r"([\d\.]+)([KM])\s*INR/(yr|month)(?:\s*-\s*([\d\.]+)([KM])\s*INR/(yr|month))?", s, flags=re.I)
    if m:
        mult = {"K": 1e3, "M": 1e6}
        f1 = 12 if m.group(3).lower() == "month" else 1
        lo = float(m.group(1)) * mult[m.group(2).upper()] * f1 / 1e5
        if m.group(4):
            f2 = 12 if m.group(6).lower() == "month" else 1
            hi = float(m.group(4)) * mult[m.group(5).upper()] * f2 / 1e5
        else:
            hi = lo
        return lo, hi, "linkedin_monthly" if f1 == 12 else "linkedin_annual"
    return "UNPARSED"

def pick(x, i):
    if not isinstance(x, tuple):
        return np.nan
    return np.nan if x[i] is None else x[i]

parsed = df["salary_portal"].map(parse_salary)
print("UNPARSED strings:", df.loc[parsed.map(lambda x: x == "UNPARSED"), "salary_portal"].unique().tolist())
df["salary_min_lpa"] = parsed.map(lambda x: pick(x, 0))
df["salary_max_lpa"] = parsed.map(lambda x: pick(x, 1))
df["salary_form"] = parsed.map(lambda x: x[2] if isinstance(x, tuple) else "")
df["salary_source"] = np.where(df["salary_form"] != "", "portal", "")

for jid, lo, hi in [("67", 1.5, 3.0), ("149", 12.0, 15.0)]:
    i = df["job_id"] == jid
    df.loc[i, ["salary_min_lpa", "salary_max_lpa"]] = [lo, hi]
    df.loc[i, "salary_form"] = "description_range"
    df.loc[i, "salary_source"] = "description"

df["salary_has"] = df["salary_min_lpa"].notna() | df["salary_max_lpa"].notna()
both = df["salary_min_lpa"].notna() & df["salary_max_lpa"].notna()
df["salary_mid_lpa"] = ((df["salary_min_lpa"] + df["salary_max_lpa"]) / 2).where(both)

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
use = ana & df["salary_has"] & ~df["is_internship"]
print("salary parsed:", int(df["salary_has"].sum()), "| analysis set:", int((ana & df["salary_has"]).sum()),
      "| usable (analysis set, not internship):", int(use.sum()))
print("usable by source:", df.loc[use, "source"].value_counts().to_dict())
print("usable by city:", df.loc[use, "primary_city"].value_counts().to_dict())
print("usable by form:", df.loc[use, "salary_form"].value_counts().to_dict())
print("usable mid (LPA) quick look:", df.loc[use, "salary_mid_lpa"].describe()[["count", "min", "50%", "max"]].round(2).to_dict())

print("--- every parsed row: raw -> min, max (LPA)")
for r in df[df["salary_has"]].itertuples():
    print(r.job_id, r.source[:3], "|", r.salary_portal[:32], "->",
          None if pd.isna(r.salary_min_lpa) else round(r.salary_min_lpa, 2),
          None if pd.isna(r.salary_max_lpa) else round(r.salary_max_lpa, 2),
          r.salary_form, "| intern" if r.is_internship else "", "| off" if r.off_target else "", "| dup" if r.is_repost_or_dup else "")

print("--- outlier candidates")
mx = df[["salary_min_lpa", "salary_max_lpa"]].max(axis=1)
mn = df[["salary_min_lpa", "salary_max_lpa"]].min(axis=1)
sus = df["job_id"].isin(["89", "99", "202", "208", "214", "237", "244"])
for r in df[df["salary_has"] & ((mx >= 15) | (mn <= 1.0) | sus)].itertuples():
    print(r.job_id, r.source[:3], r.job_title[:30], "|", r.salary_portal[:30], "| exp desc/portal:", r.exp_min, r.exp_portal_min,
          "| intern" if r.is_internship else "", "| off" if r.off_target else "")

print("--- duplicate groups: salary lost or conflicting")
for n, sub in df[df["dup_group"].notna()].groupby("dup_group"):
    kept = sub[sub["job_id"] == sub["dup_kept_job_id"].iloc[0]].iloc[0]
    others = sub[sub["job_id"] != kept["job_id"]]
    if (not kept["salary_has"]) and others["salary_has"].any():
        print("group", n, "kept", kept["job_id"], "has no salary; dropped rows with salary:", others.loc[others["salary_has"], "job_id"].tolist())
    vals = {(r.salary_min_lpa, r.salary_max_lpa) for r in sub[sub["salary_has"]].itertuples()}
    if len(vals) > 1:
        print("group", n, "different salaries:", vals)

print("--- bare-number salary mentions in descriptions (analysis set, no salary yet)")
sw = re.compile(r"(?:salary|ctc|compensation|per month|per annum|p\.a\.)[^.\n]{0,25}\d", re.I)
k = 0
for jid, t in zip(df.loc[ana & ~df["salary_has"], "job_id"], df.loc[ana & ~df["salary_has"], "description_text"]):
    m = sw.search(t)
    if m:
        k += 1
        if k <= 20:
            print(jid, repr(re.sub(r"\s+", " ", t[max(0, m.start() - 20): m.end() + 25])))
print("rows with such a mention:", k)

df.drop(columns=["_cands", "_excl"], errors="ignore").to_csv("../data_clean/stage11_salary.csv", index=False, encoding="utf-8")
chk = pd.read_csv("../data_clean/stage11_salary.csv", dtype=str, keep_default_na=False)
print(chk.shape, chk["job_id"].is_unique)

In [ ]:
reasons = {
 "high_pay_10plus_lpa": ["27", "28", "99", "208", "244"],
 "high_for_fresher_title": ["89"],
 "minimum_below_1_lpa": ["48", "57", "58"],
 "possible_platform_estimate": ["202", "237"],
 "very_wide_range": ["214"],
}
rcode = {j: r for r, ids in reasons.items() for j in ids}
df["salary_outlier"] = df["job_id"].isin(rcode)
df["salary_outlier_reason"] = df["job_id"].map(rcode).fillna("")

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
df["salary_usable"] = ana & df["salary_has"] & ~df["is_internship"]
df["salary_usable_clean"] = df["salary_usable"] & ~df["salary_outlier"]

print("usable:", int(df["salary_usable"].sum()), "| flagged outliers among them:",
      int((df["salary_usable"] & df["salary_outlier"]).sum()), "| usable_clean:", int(df["salary_usable_clean"].sum()))
for name, m in [("usable", df["salary_usable"]), ("usable_clean", df["salary_usable_clean"])]:
    s = df.loc[m, "salary_mid_lpa"].dropna()
    print(name, "| n with midpoint:", len(s), "| p25/median/p75:", [round(x, 2) for x in s.quantile([0.25, 0.5, 0.75])])
print("usable_clean by source:", df.loc[df["salary_usable_clean"], "source"].value_counts().to_dict())
print("usable_clean by city:", df.loc[df["salary_usable_clean"], "primary_city"].value_counts().to_dict())

df.drop(columns=["_cands", "_excl"], errors="ignore").to_csv("../data_clean/stage11_salary.csv", index=False, encoding="utf-8")
chk = pd.read_csv("../data_clean/stage11_salary.csv", dtype=str, keep_default_na=False)
print(chk.shape, chk["job_id"].is_unique)

In [ ]:
from collections import Counter

CS = 0
SKILLS = {
 "SQL": (r"\bSQL\b", CS),
 "MySQL": (r"\bMy ?SQL\b", re.I),
 "PostgreSQL": (r"\bPostgre(?:s|SQL)\w*\b", re.I),
 "SQL Server": (r"\b(?:MS |Microsoft )?SQL Server\b|\bMSSQL\b|\bT-SQL\b", re.I),
 "Oracle": (r"\bOracle\b", CS),
 "Excel": (r"\b(?:Microsoft |MS )?Excel\b", CS),
 "Advanced Excel": (r"\bAdvanced (?:MS )?Excel\b", re.I),
 "VBA/Macros": (r"\bVBA\b|\bmacros?\b", re.I),
 "Power BI": (r"\bPower ?BI\b", re.I),
 "Power Query": (r"\bPower Query\b", re.I),
 "DAX": (r"\bDAX\b", CS),
 "Tableau": (r"\bTableau\b", re.I),
 "Looker": (r"\bLooker(?: Studio)?\b", re.I),
 "Qlik": (r"\bQlik\w*\b", re.I),
 "Google Data Studio": (r"\bData Studio\b", re.I),
 "Google Sheets": (r"\bGoogle Sheets?\b", re.I),
 "Google Analytics": (r"\bGoogle Analytics\b|\bGA4\b", re.I),
 "Python": (r"\bPython\b", re.I),
 "R": (r"(?<![A-Za-z0-9&/.\-])R(?![A-Za-z0-9&/\-])", CS),
 "SAS": (r"\bSAS\b", CS),
 "SPSS": (r"\bSPSS\b", CS),
 "Snowflake": (r"\bSnowflake\b", re.I),
 "BigQuery": (r"\bBig ?Query\b", re.I),
 "Redshift": (r"\bRedshift\b", re.I),
 "Databricks": (r"\bDatabricks\b", re.I),
 "Spark": (r"\b(?:Py)?Spark\b", re.I),
 "Hadoop": (r"\bHadoop\b|\bHive\b", re.I),
 "AWS": (r"\bAWS\b|\bAmazon Web Services\b", CS),
 "Azure": (r"\bAzure\b", re.I),
 "GCP": (r"\bGCP\b|\bGoogle Cloud\b", CS),
 "ETL": (r"\bETL\b", CS),
 "Alteryx": (r"\bAlteryx\b", re.I),
 "SAP": (r"\bSAP\b", CS),
 "Salesforce": (r"\bSalesforce\b", re.I),
 "Jira": (r"\bJIRA\b|\bJira\b", CS),
 "Pivot tables": (r"\bpivot(?: tables?)?\b", re.I),
 "VLOOKUP/XLOOKUP": (r"\b[VHX]LOOKUP\b|\bINDEX[- ]?MATCH\b", re.I),
 "Statistics": (r"\bstatistic\w*\b", re.I),
 "A/B testing": (r"\bA/B test\w*\b", re.I),
 "Machine learning": (r"\bmachine learning\b|\bML\b", CS),
 "Data visualization": (r"\bdata visuali[sz]ation\b|\bdashboards?\b", re.I),
}
comp = {k: re.compile(p, f) for k, (p, f) in SKILLS.items()}

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
A = df[ana]
rows = []
for k, rx in comp.items():
    d = A["description_clean"].map(lambda t: bool(rx.search(t)))
    p = A["skills_portal"].map(lambda t: bool(rx.search(t)))
    rows.append((k, int(d.sum()), int(p.sum())))
tab = pd.DataFrame(rows, columns=["skill", "in_description", "in_portal_tags"]).sort_values("in_description", ascending=False)
print("analysis set rows:", len(A), "| rows with portal tags:", int((A["skills_portal"] != "No tags").sum()))
print(tab.to_string(index=False))

anyhit = A["description_clean"].map(lambda t: any(rx.search(t) for rx in comp.values()))
print("rows with none of the starter skills:", int((~anyhit).sum()), "| by source:", A.loc[~anyhit, "source"].value_counts().to_dict())

print("--- most common portal tags NOT covered by the starter list (analysis set)")
cnt = Counter()
for t in A.loc[A["skills_portal"] != "No tags", "skills_portal"]:
    for tag in [x.strip() for x in t.split(",")]:
        if tag and not any(rx.search(tag) for rx in comp.values()):
            cnt[tag.lower()] += 1
for tag, n in cnt.most_common(50):
    print(n, "|", tag[:50])

print("--- collision check: 'R' matches (first 15 snippets)")
k = 0
for jid, t in zip(A["job_id"], A["description_clean"]):
    m = comp["R"].search(t)
    if m and k < 15:
        k += 1
        print(jid, repr(re.sub(r"\s+", " ", t[max(0, m.start() - 30): m.end() + 30])))
print("rows with an 'R' match:", int(A["description_clean"].map(lambda t: bool(comp["R"].search(t))).sum()))

print("--- collision check: lowercase 'excel' as a verb (first 10)")
k = 0
verb = re.compile(r"\bexcel(?:s|led)? (?:in|at|with|as|on)\b|\bexcelling\b", re.I)
for jid, t in zip(A["job_id"], A["description_clean"]):
    m = verb.search(t)
    if m and k < 10:
        k += 1
        print(jid, repr(re.sub(r"\s+", " ", t[max(0, m.start() - 30): m.end() + 30])))
print("--- SQL near Sales Qualified (job 268):",
      bool(re.search(r"Sales Qualified", df.loc[df["job_id"] == "268", "description_clean"].iloc[0], re.I)))

In [ ]:
import re, numpy as np
from collections import Counter

I = re.I
VOC = {
 "SQL": (r"\bSQL\b", I),
 "MySQL": (r"\bMy ?SQL\b", I),
 "PostgreSQL": (r"\bPostgre(?:s|SQL)\w*\b", I),
 "SQL Server": (r"\b(?:MS |Microsoft )?SQL Server\b|\bMSSQL\b|\bT-SQL\b", I),
 "Oracle": (r"\bOracle\b", 0),
 "Excel": (r"\b(?:Microsoft |MS )?Excel\b(?! in (?:a |an |the )?(?:fast|dynamic|high|this|our)\b)(?! at\b)", 0),
 "Excel formulas": (r"\b(?:[VHX]LOOKUP|SUMIFS?|COUNTIFS?|IFERROR)\b|\bINDEX[- ]?MATCH\b|\bconditional formatting\b", I),
 "Pivot tables": (r"\bpivot ?(?:tables?|charts?)\b", I),
 "VBA/Macros": (r"\bVBA\b|\bmacros\b|\bmacro (?:programming|automation)\b", I),
 "PowerPoint": (r"\bPower ?Point\b|\bPPT\b", I),
 "Power BI": (r"\bPower ?BI\b", I),
 "Power Query": (r"\bPower Query\b", I),
 "DAX": (r"\bDAX\b", 0),
 "Tableau": (r"\bTableau\b", I),
 "Looker": (r"\bLooker\b(?! Studio)", I),
 "Looker Studio": (r"\bLooker Studio\b|\bData Studio\b", I),
 "Qlik": (r"\bQlik\w*\b", I),
 "Google Sheets": (r"\bGoogle Sheets?\b", I),
 "Google Analytics": (r"\bGoogle Analytics\b|\bGA4\b", I),
 "Python": (r"\bPython\b", I),
 "R": (r"(?<![A-Za-z0-9&/.\-])R(?![A-Za-z0-9&/\-])", 0),
 "SAS": (r"\bSAS\b", 0),
 "SPSS": (r"\bSPSS\b", 0),
 "Snowflake": (r"\bSnowflake\b", I),
 "BigQuery": (r"\bBig ?Query\b", I),
 "Redshift": (r"\bRedshift\b", I),
 "Databricks": (r"\bDatabricks\b", I),
 "Spark": (r"\b(?:Py)?Spark\b", 0),
 "Hadoop/Hive": (r"\bHadoop\b|\bHive\b", 0),
 "AWS": (r"\bAWS\b|\bAmazon Web Services\b", 0),
 "Azure": (r"\bAzure\b", I),
 "GCP": (r"\bGCP\b|\bGoogle Cloud\b", 0),
 "ETL": (r"\bETL\b", I),
 "Alteryx": (r"\bAlteryx\b", I),
 "SAP": (r"\bSAP\b", 0),
 "Salesforce": (r"\bSalesforce\b", I),
 "Jira": (r"\bJIRA\b|\bJira\b", 0),
 "Statistics": (r"\bstatistic\w*\b", I),
 "A/B testing": (r"\bA/B test\w*\b", I),
 "Machine learning": (r"\bmachine learning\b|\bML\b", 0),
 "Dashboards/visualization": (r"\bdata visuali[sz]ation\b|\bdashboards?\b", I),
 "Data cleaning/quality": (r"\bdata (?:cleaning|cleansing|validation|quality|integrity)\b", I),
 "Data modeling": (r"\bdata model(?:l)?ing\b", I),
 "Data warehousing": (r"\bdata warehous\w+\b|\bDWH\b", I),
 "Requirement gathering": (r"\brequirements? (?:gathering|analysis|elicitation)\b|\bBRDs?\b|\bFRDs?\b", I),
}
comp = {k: re.compile(p, f) for k, (p, f) in VOC.items()}
comp_tag = {k: re.compile(p, I) for k, (p, f) in VOC.items()}

PREF_CUE = re.compile(r"\bpreferred\b(?! (?:candidate|location|industry|shift|notice))|\ba plus\b|\badded advantage\b|\ban advantage\b|\bnice to have\b|\bgood to have\b|\bdesirable\b|\bbonus\b|\boptional\b|\bnot mandatory\b|\bnot required\b|\bbeneficial\b|\ban asset\b|\bplus point", I)
HEAD_PREF = re.compile(r"preferred (?:skills?|qualifications?|experience|requirements?)|nice to have|good to have|bonus|desirable|added advantage|plus points?|what would set you apart", I)
KEYSKILLS_HEAD = re.compile(r"^\s*key skills?\b", I)
SKILLS_INLINE = re.compile(r"^\s*skills\s*:", I)
SENT = re.compile(r"(?<=[.;!?])\s+")
BULLET = ("\u2022", "-", "*", "\u00b7", "\u2013", "\u2014", "\u25cf", "\u25aa")
SQL_BAD = re.compile(r"\bSAL\b|\bMQL\b|\bSQO\b|\bCQL\b")
RANK = {"keyskills": 1, "preferred": 2, "stated": 3}
sql_rej = []
log268 = []

def is_heading(s):
    return len(s) <= 50 and not s.startswith(BULLET) and not s.endswith((".", ";")) and "," not in s and len(s.split()) <= 6

def desc_levels(text, jid):
    best = {}
    in_pref, in_key = False, False
    for line in text.split("\n"):
        s = line.strip()
        if not s:
            continue
        hits = [(sk, m) for sk, rx in comp.items() for m in rx.finditer(s)]
        if is_heading(s) and not hits:
            in_pref = bool(HEAD_PREF.search(s))
            in_key = bool(KEYSKILLS_HEAD.match(s))
            continue
        inline_key = bool(KEYSKILLS_HEAD.match(s) or SKILLS_INLINE.match(s))
        pos = 0
        for sent in SENT.split(s):
            idx = s.find(sent, pos)
            pos = idx + len(sent)
            sh = [(sk, m) for sk, m in hits if idx <= m.start() < idx + len(sent)]
            if not sh:
                continue
            pref = in_pref or bool(PREF_CUE.search(sent))
            for sk, m in sh:
                if sk == "SQL" and SQL_BAD.search(s[max(0, m.start() - 30): m.end() + 30]):
                    sql_rej.append(jid)
                    if jid == "268":
                        log268.append(("rejected", s[max(0, m.start() - 30): m.end() + 30]))
                    continue
                if sk == "SQL" and jid == "268":
                    log268.append(("kept", s[max(0, m.start() - 30): m.end() + 30]))
                lvl = "preferred" if pref else ("keyskills" if (in_key or inline_key) else "stated")
                if sk not in best:
                    best[sk] = [lvl, 1]
                else:
                    best[sk][1] += 1
                    if RANK[lvl] > RANK[best[sk][0]]:
                        best[sk][0] = lvl
    return best

def split_tags(s):
    if s in ("No tags", ""):
        return []
    out, depth, cur = [], 0, ""
    for ch in s:
        if ch == "(":
            depth += 1
        if ch == ")":
            depth = max(0, depth - 1)
        if ch == "," and depth == 0:
            out.append(cur.strip()); cur = ""
        else:
            cur += ch
    out.append(cur.strip())
    return [x for x in out if x]

def tag_skills(s):
    out = set()
    for tg in split_tags(s):
        for sk, rx in comp_tag.items():
            if sk == "R":
                if tg.strip().lower() == "r":
                    out.add("R")
            elif rx.search(tg):
                out.add(sk)
    return out

rows = []
for r in df.itertuples():
    lv = desc_levels(r.description_clean, r.job_id)
    tg = tag_skills(r.skills_portal)
    for sk in set(lv) | tg:
        rows.append((r.job_id, sk, lv[sk][0] if sk in lv else "", lv[sk][1] if sk in lv else 0, sk in tg))
js = pd.DataFrame(rows, columns=["job_id", "skill", "desc_level", "desc_mentions", "in_portal_tags"])
df["excel_advanced"] = df["description_clean"].str.contains(r"\bAdvanced (?:MS |Microsoft )?Excel\b", case=False, regex=True)

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
aid = set(df.loc[ana, "job_id"])
J = js[js["job_id"].isin(aid)].copy()
J["any_desc"] = J["desc_level"] != ""
J["stated"] = J["desc_level"] == "stated"
J["preferred"] = J["desc_level"] == "preferred"
J["keyskills"] = J["desc_level"] == "keyskills"
J["tag_not_in_desc"] = (J["desc_level"] == "") & J["in_portal_tags"]
summ = J.groupby("skill")[["any_desc", "stated", "preferred", "keyskills", "in_portal_tags", "tag_not_in_desc"]].sum().astype(int)
print("analysis rows:", len(aid), "| rows with portal tags:", int((ana & (df["skills_portal"] != "No tags")).sum()))
print(summ.sort_values("any_desc", ascending=False).to_string())

print("--- SQL rejected as Sales Qualified Lead:", dict(Counter(sql_rej)))
print("--- job 268 SQL decisions")
for d, s in log268:
    print(d, repr(re.sub(r"\s+", " ", s)))

old_excel = re.compile(r"\b(?:Microsoft |MS )?Excel\b")
lost = [j for j, t in zip(df["job_id"], df["description_clean"])
        if old_excel.search(t) and not (js[(js["job_id"] == j) & (js["skill"] == "Excel") & (js["desc_level"] != "")].shape[0])]
print("--- rows where Excel was matched before but not now:", lost)

seen = {"16", "25", "29", "32", "35", "36", "45", "52", "66", "117", "122", "128", "135", "136", "169"}
print("--- remaining 'R' matches (analysis set)")
for jid, t in zip(df["job_id"], df["description_clean"]):
    if jid in aid and jid not in seen:
        m = comp["R"].search(t)
        if m:
            print(jid, repr(re.sub(r"\s+", " ", t[max(0, m.start() - 30): m.end() + 30])))

print("--- rows with a keyskills-level skill:", sorted(set(J.loc[J["keyskills"], "job_id"]), key=int)[:30])
n = J[J["any_desc"]].groupby("job_id")["skill"].nunique().sort_values(ascending=False).head(12)
print("--- most distinct skills in one posting:", {k: int(v) for k, v in n.items()})
per = J[J["any_desc"]].groupby("job_id")["skill"].nunique()
print("median skills per row (description, rows with 1+):", float(per.median()), "| rows with none:", len(aid) - len(per))
none_ids = aid - set(per.index)
print("rows with no description skill, by source:", df[df["job_id"].isin(none_ids)]["source"].value_counts().to_dict())

tg = J[J["in_portal_tags"]]
print("--- tag-vs-description agreement: tag pairs", len(tg), "| also in description", int(tg["any_desc"].sum()))

js.to_csv("../data_clean/stage12_job_skills.csv", index=False, encoding="utf-8")
print(js.shape, "| distinct skills:", js["skill"].nunique())

In [ ]:
from collections import defaultdict

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
A = df[ana]
have = set(zip(js.loc[js["desc_level"] != "", "job_id"], js.loc[js["desc_level"] != "", "skill"]))

tokens = {"sql": "SQL", "excel": "Excel", "python": "Python", "tableau": "Tableau", "looker": "Looker",
          "snowflake": "Snowflake", "azure": "Azure", "bigquery": "BigQuery", "databricks": "Databricks",
          "salesforce": "Salesforce", "qlik": "Qlik", "powerbi": "Power BI"}
legit = {"mysql", "postgresql", "nosql", "sqlite", "sqlalchemy", "plsql", "tsql", "mssql",
         "excellent", "excellence", "excelled", "excels", "excelling", "excellently"}

word_rows = defaultdict(set)
missed = defaultdict(set)
for jid, t in zip(A["job_id"], A["description_clean"]):
    for w in re.findall(r"[A-Za-z]+", t):
        wl = w.lower()
        for tok, sk in tokens.items():
            if tok in wl and wl != tok and wl not in legit:
                word_rows[w].add(jid)
                if (jid, sk) not in have:
                    missed[sk].add(jid)

print("--- words containing a skill name but not equal to it (analysis set), by number of rows")
for w, ids in sorted(word_rows.items(), key=lambda x: -len(x[1]))[:35]:
    print(len(ids), "|", w)
print("--- skills that appear ONLY inside such a word (missed by word-boundary matching):", {k: len(v) for k, v in missed.items()})
print("rows affected:", len(set().union(*missed.values())) if missed else 0)

DEG = re.compile(r"degree|bachelor|master|b\.?\s?tech|m\.?\s?tech|\bb\.?e\b|graduate|\bmba\b|diploma|field of study|qualification", re.I)
for name in ["Statistics", "Machine learning"]:
    rx = comp[name]
    n_any = n_deg_only = 0
    ex = []
    for jid, t in zip(A["job_id"], A["description_clean"]):
        kinds = ["degree" if DEG.search(t[max(0, m.start() - 90): m.end() + 90]) else "skill" for m in rx.finditer(t)]
        if kinds:
            n_any += 1
            if all(k == "degree" for k in kinds):
                n_deg_only += 1
                if len(ex) < 6:
                    m = rx.search(t)
                    ex.append((jid, re.sub(r"\s+", " ", t[max(0, m.start() - 60): m.end() + 40])))
    print("---", name, "| rows with a mention:", n_any, "| every mention is next to degree wording:", n_deg_only)
    for jid, s in ex:
        print("   ", jid, repr(s))

pref = js[(js["desc_level"] == "preferred") & js["job_id"].isin(set(A["job_id"]))]
print("--- 12 random 'preferred' decisions (cue = what triggered it)")
smp = pref.sample(12, random_state=5)
for jid, sk in zip(smp["job_id"], smp["skill"]):
    t = df.loc[df["job_id"] == jid, "description_clean"].iloc[0]
    for line in t.split("\n"):
        m = comp[sk].search(line)
        if m:
            cue = "sentence cue" if PREF_CUE.search(line) else "heading or section"
            print(jid, sk, "|", cue, "|", repr(re.sub(r"\s+", " ", line[max(0, m.start() - 50): m.end() + 50])))
            break

In [ ]:
FIELDS = re.compile(r"mathematics|maths|economics|computer science|engineering|data science|information technology|physics|operations research|commerce|\bHR\b", re.I)
DEGW = re.compile(r"degree|bachelor|master|b\.?\s?tech|m\.?\s?tech|\bb\.?e\b|graduate|postgraduate|\bmba\b|diploma|qualification|field", re.I)

def stat_is_degree(t, m):
    s0 = max(t.rfind(".", 0, m.start()), t.rfind("\n", 0, m.start())) + 1
    left = t[s0:m.start()]
    near = t[max(0, m.start() - 40): m.start()] + " " + t[m.end(): m.end() + 40]
    return bool(DEGW.search(left[-60:])) or (bool(FIELDS.search(near)) and bool(DEGW.search(t[s0: m.end() + 60])))

rx = comp["Statistics"]
new_skill, new_deg, recovered = [], [], []
for jid, t in zip(A["job_id"], A["description_clean"]):
    ms = list(rx.finditer(t))
    if not ms:
        continue
    kinds = ["degree" if stat_is_degree(t, m) else "skill" for m in ms]
    old_kinds = ["degree" if DEG.search(t[max(0, m.start() - 90): m.end() + 90]) else "skill" for m in ms]
    if "skill" in kinds:
        new_skill.append(jid)
        if all(k == "degree" for k in old_kinds):
            recovered.append((jid, re.sub(r"\s+", " ", t[max(0, ms[0].start() - 55): ms[0].end() + 40])))
    else:
        new_deg.append((jid, re.sub(r"\s+", " ", t[max(0, ms[0].start() - 55): ms[0].end() + 40])))

print("Statistics, analysis set: rows with a mention", len(new_skill) + len(new_deg),
      "| new rule: skill", len(new_skill), "| degree-only", len(new_deg), "| old rule degree-only: 34")
print("--- degree-only under the new rule (first 8)")
for jid, s in new_deg[:8]:
    print("   ", jid, repr(s))
print("--- recovered (old rule said degree-only, new rule says skill), first 8")
for jid, s in recovered[:8]:
    print("   ", jid, repr(s))

print("--- headings that switch on 'preferred' mode (analysis set)")
hc = Counter()
for t in A["description_clean"]:
    for line in t.split("\n"):
        s = line.strip()
        if s and is_heading(s) and HEAD_PREF.search(s) and not any(r.search(s) for r in comp.values()):
            hc[s.lower()] += 1
for h, n in hc.most_common(30):
    print(n, "|", h)

def nearest_heading(t, pos):
    for s in reversed(t[:pos].split("\n")[:-1]):
        s = s.strip()
        if s and is_heading(s):
            return s
    return ""

print("--- nearest heading above the skill, jobs 105 and 283")
for jid, sk in [("105", "Tableau"), ("283", "Python")]:
    t = df.loc[df["job_id"] == jid, "description_clean"].iloc[0]
    m = comp[sk].search(t)
    print(jid, sk, "| heading:", repr(nearest_heading(t, m.start())), "| in HEAD_PREF:", bool(HEAD_PREF.search(nearest_heading(t, m.start()))))

In [ ]:
old_js = js.copy()

HEAD_PREF = re.compile(r"^(?!.*:\s*(?:na|n/a|none)\s*$)(?=.*(?:preferred (?:skills?|qualifications?|experience|requirements?)|nice to have|good to have|desirable|bonus(?! (?:incentive|pay|payout|structure|scheme))|added advantage|plus points?|what would set you apart))", re.I)

TOOLS = ["SQL", "MySQL", "PostgreSQL", "SQL Server", "Oracle", "Excel", "VBA/Macros", "PowerPoint", "Power BI",
         "Power Query", "DAX", "Tableau", "Looker", "Looker Studio", "Qlik", "Google Sheets", "Google Analytics",
         "Python", "R", "SAS", "SPSS", "Snowflake", "BigQuery", "Redshift", "Databricks", "Spark", "Hadoop/Hive",
         "AWS", "Azure", "GCP", "Alteryx", "SAP", "Salesforce", "Jira"]
type_map = {sk: ("tool" if sk in TOOLS else "concept") for sk in VOC}
STAT_NOT_SKILL = {"301"}

sql_rej.clear()
log268.clear()
rows = []
for r in df.itertuples():
    lv = desc_levels(r.description_clean, r.job_id)
    tg = tag_skills(r.skills_portal)
    if "Statistics" in lv:
        ms = list(comp["Statistics"].finditer(r.description_clean))
        if r.job_id in STAT_NOT_SKILL or all(stat_is_degree(r.description_clean, m) for m in ms):
            del lv["Statistics"]
    for sk in set(lv) | tg:
        rows.append((r.job_id, sk, lv[sk][0] if sk in lv else "", lv[sk][1] if sk in lv else 0, sk in tg))
js = pd.DataFrame(rows, columns=["job_id", "skill", "desc_level", "desc_mentions", "in_portal_tags"])
js["skill_type"] = js["skill"].map(type_map)

m = old_js.merge(js, on=["job_id", "skill"], how="outer", suffixes=("_old", "_new"))
chg = m[m["desc_level_old"].fillna("") != m["desc_level_new"].fillna("")]
print("(job, skill) rows whose description level changed:", len(chg))
for r in chg.head(30).itertuples():
    print("  ", r.job_id, r.skill, repr(r.desc_level_old), "->", repr(r.desc_level_new))

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
J = js[js["job_id"].isin(set(df.loc[ana, "job_id"]))].copy()
J["any_desc"] = J["desc_level"] != ""
J["stated"] = J["desc_level"] == "stated"
J["preferred"] = J["desc_level"] == "preferred"
for typ, n in [("tool", 18), ("concept", 12)]:
    s = (J[J["skill_type"] == typ].groupby("skill")[["any_desc", "stated", "preferred", "in_portal_tags"]]
         .sum().astype(int).sort_values("any_desc", ascending=False).head(n))
    print("---", typ, "(analysis set, 246 rows)")
    print(s.to_string())
print("Statistics, analysis set rows:", int(J[(J["skill"] == "Statistics") & J["any_desc"]].shape[0]))

df["da_title"] = df["job_title"].str.contains(r"data analy", case=False, regex=True)
def band(x):
    if pd.isna(x):
        return "unknown"
    return "fresher (0-1)" if x <= 1 else "experienced (2+)"
df["exp_band_desc"] = df["exp_min"].map(band)
df["exp_band_fallback"] = df["exp_min_fallback"].map(band)
g = ana & df["da_title"]
print("--- Q1 base: Data Analyst-titled rows in the analysis set:", int(g.sum()), "| by source:", df.loc[g, "source"].value_counts().to_dict())
print("band by description:", df.loc[g, "exp_band_desc"].value_counts().to_dict())
print("band with portal fallback:", df.loc[g, "exp_band_fallback"].value_counts().to_dict())
print("by source and description band:")
print(pd.crosstab(df.loc[g, "source"], df.loc[g, "exp_band_desc"]).to_string())

skills_dim = pd.DataFrame({"skill": list(VOC), "skill_type": [type_map[s] for s in VOC]})
js.to_csv("../data_clean/stage12_job_skills.csv", index=False, encoding="utf-8")
skills_dim.to_csv("../data_clean/stage12_skills.csv", index=False, encoding="utf-8")
df.drop(columns=["_cands", "_excl"], errors="ignore").to_csv("../data_clean/stage12_jobs.csv", index=False, encoding="utf-8")
print(js.shape, "| distinct skills:", js["skill"].nunique(), "| skills table:", skills_dim.shape)

In [ ]:
chg_rest = chg.iloc[30:]
aid = set(df.loc[ana, "job_id"])
print("--- the other changed rows:", len(chg_rest))
for r in chg_rest.itertuples():
    print("  ", r.job_id, r.skill, repr(r.desc_level_old), "->", repr(r.desc_level_new), "| analysis set:", r.job_id in aid)

rest_tools = (J[J["skill_type"] == "tool"].groupby("skill")[["any_desc", "stated", "preferred", "in_portal_tags"]]
              .sum().astype(int).sort_values("any_desc", ascending=False).iloc[18:])
print("--- remaining tools (analysis set)")
print(rest_tools.to_string())

def cat(s):
    if re.fullmatch(r"\s*data analyst\s*", s): return "exactly 'Data Analyst'"
    if re.search(r"\bsenior\b|\bsr\b|\blead\b|\bprincipal\b|\bmanager\b", s): return "senior/lead"
    if re.search(r"associate", s): return "associate"
    if re.search(r"junior|\bjr\b|fresher|trainee|intern", s): return "junior/fresher/trainee/intern"
    if re.search(r"data analytic", s): return "'data analytics ...' titles"
    return "other data analyst variants"

sub = df.loc[g, ["job_id", "job_title", "source"]].copy()
sub["cat"] = sub["job_title"].str.lower().map(cat)
print("--- Q1 base title categories:", sub["cat"].value_counts().to_dict())
for c in ["senior/lead", "associate", "'data analytics ...' titles", "other data analyst variants"]:
    print("---", c)
    for jid, ti, sr in zip(sub.loc[sub["cat"] == c, "job_id"], sub.loc[sub["cat"] == c, "job_title"], sub.loc[sub["cat"] == c, "source"]):
        print("  ", jid, sr[:3], "|", ti[:60])

In [ ]:
ana = ~df["off_target"] & ~df["is_repost_or_dup"]
A = df[ana]
t = df["job_title"].str.lower().str.replace("_", " ", regex=False)
has_da = t.str.contains("data analyst", regex=False)
assoc = t.str.contains("associate", regex=False)
two_role = df["job_id"].isin(["34", "35", "256"])
df["q1_base"] = ana & has_da & ~assoc & ~two_role
df["q1_assoc_sens"] = ana & has_da & assoc & ~two_role

old = ana & df["da_title"]
print("old loose base:", int(old.sum()), "| new q1_base:", int(df["q1_base"].sum()),
      "| associate sensitivity rows:", int(df["q1_assoc_sens"].sum()))
print("removed, no 'data analyst' in title:", df.loc[old & ~has_da, "job_id"].tolist())
print("removed, Associate:", df.loc[old & has_da & assoc & ~two_role, "job_id"].tolist())
print("removed, two-role title:", df.loc[old & two_role, "job_id"].tolist())

b = df["q1_base"]
print("q1_base by source:", df.loc[b, "source"].value_counts().to_dict())
print("band by description:", df.loc[b, "exp_band_desc"].value_counts().to_dict())
print(pd.crosstab(df.loc[b, "source"], df.loc[b, "exp_band_desc"]).to_string())
print("band with portal fallback:", df.loc[b, "exp_band_fallback"].value_counts().to_dict())
print("internships inside q1_base:", df.loc[b & df["is_internship"], "job_id"].tolist())
print("senior/lead titles inside q1_base:", df.loc[b & t.str.contains(r"senior|\bsr\b|lead|principal|manager", regex=True), "job_id"].tolist())
print("borderline rows inside q1_base:", int((b & df["borderline"]).sum()))

print("--- Spark and Hadoop/Hive snippets (analysis set)")
for sk in ["Spark", "Hadoop/Hive"]:
    k = 0
    for jid, tx in zip(A["job_id"], A["description_clean"]):
        m = comp[sk].search(tx)
        if m and k < 8:
            k += 1
            print(sk, jid, repr(re.sub(r"\s+", " ", tx[max(0, m.start() - 35): m.end() + 35])))

df.drop(columns=["_cands", "_excl"], errors="ignore").to_csv("../data_clean/stage12_jobs.csv", index=False, encoding="utf-8")
print("saved", df.shape)

In [ ]:
mask_sp = (js["job_id"] == "192") & (js["skill"] == "Spark")
mask_hv = (js["job_id"] == "190") & (js["skill"] == "Hadoop/Hive")
print(js[mask_sp | mask_hv][["job_id", "skill", "desc_level", "desc_mentions", "in_portal_tags"]].to_string(index=False))
targets = list(js.index[mask_sp | mask_hv])
for i in targets:
    if js.at[i, "desc_mentions"] == 1:
        if js.at[i, "in_portal_tags"]:
            js.at[i, "desc_level"] = ""
            js.at[i, "desc_mentions"] = 0
        else:
            js = js.drop(i)
        print("patched", i)
    else:
        print("NOT patched (more than one mention):", i)

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
A = df[ana]
ana_ids = set(A["job_id"])
J2 = js[js["job_id"].isin(ana_ids) & (js["desc_level"] != "")]
print("analysis set rows now: Spark", int((J2["skill"] == "Spark").sum()), "| Hadoop/Hive", int((J2["skill"] == "Hadoop/Hive").sum()))

seen_sp = {"21", "110", "134", "147", "162", "178", "185", "192"}
print("--- Spark rows not seen before")
for jid, tx in zip(A["job_id"], A["description_clean"]):
    if jid not in seen_sp:
        m = comp["Spark"].search(tx)
        if m:
            print("Spark", jid, repr(re.sub(r"\s+", " ", tx[max(0, m.start() - 35): m.end() + 35])))
js.to_csv("../data_clean/stage12_job_skills.csv", index=False, encoding="utf-8")

print("--- portal education (analysis set)")
pe = A["education"].value_counts()
print("Not mentioned:", int(pe.get("Not mentioned", 0)), "| distinct values:", len(pe))
for v, n in pe.drop("Not mentioned", errors="ignore").head(12).items():
    print(n, "|", v[:80])

EDU = {
 "any graduate / any degree": re.compile(r"\bany (?:graduate|graduation|degree|bachelor\w*|discipline|stream)\b|\bgraduate in any\b|\bdegree in any\b", re.I),
 "bachelor's or graduate (generic)": re.compile(r"\bbachelor[\u2019']?s?\b|\bgraduat(?:e|es|ion)\b|\bundergraduate\b|\bdegree\b", re.I),
 "engineering (B.Tech / B.E.)": re.compile(r"\bb\.?\s?tech\b|bachelor of engineering|engineering degree|\bB\.\s?E\b\.?", re.I),
 "computer applications (BCA / MCA)": re.compile(r"\bbca\b|\bmca\b|computer applications", re.I),
 "science (B.Sc / M.Sc)": re.compile(r"\bb\.?\s?sc\b|\bm\.?\s?sc\b|bachelor of science|master of science", re.I),
 "commerce (B.Com / M.Com)": re.compile(r"\bb\.?\s?com\b|\bm\.?\s?com\b|bachelor of commerce", re.I),
 "management (BBA / MBA / PGDM)": re.compile(r"\bbba\b|\bmba\b|\bpgdm\b|\bbms\b", re.I),
 "master's or postgraduate": re.compile(r"\bmaster[\u2019']s\b|\bmasters? degree\b|\bmaster of\b|\bpostgraduate\b|\bpost[- ]graduate\b|\bm\.?\s?tech\b", re.I),
 "PhD / doctorate": re.compile(r"\bph\.?\s?d\b|\bdoctorate\b|\bdoctoral\b", re.I),
 "diploma": re.compile(r"\bdiploma\b", re.I),
 "degree not required / any background": re.compile(r"\bdegree (?:is )?not (?:required|mandatory)\b|\bno degree\b|\bgraduation not required\b|\bwithout a degree\b|\ball backgrounds\b|\bany background\b", re.I),
}
print("--- education words in descriptions (analysis set, rows with at least one mention)")
anyedu = pd.Series(False, index=A.index)
for name, rx in EDU.items():
    hit = A["description_clean"].map(lambda tx: bool(rx.search(tx)))
    anyedu |= hit
    print(int(hit.sum()), "|", name)
    k = 0
    for jid, tx in zip(A.loc[hit, "job_id"], A.loc[hit, "description_clean"]):
        if k < 2:
            m = rx.search(tx)
            print("      ", jid, repr(re.sub(r"\s+", " ", tx[max(0, m.start() - 45): m.end() + 45])))
            k += 1
print("rows with no education word at all:", int((~anyedu).sum()), "| by source:", A.loc[~anyedu, "source"].value_counts().to_dict())

In [ ]:
mask = (js["job_id"] == "265") & (js["skill"] == "Spark")
print(js[mask][["job_id", "skill", "desc_level", "desc_mentions", "in_portal_tags"]].to_string(index=False))
for i in list(js.index[mask]):
    if js.at[i, "desc_mentions"] == 1:
        if js.at[i, "in_portal_tags"]:
            js.at[i, "desc_level"] = ""
            js.at[i, "desc_mentions"] = 0
        else:
            js = js.drop(i)
        print("patched", i)
    else:
        print("NOT patched (more than one mention):", i)
js.to_csv("../data_clean/stage12_job_skills.csv", index=False, encoding="utf-8")

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
J2 = js[js["job_id"].isin(set(df.loc[ana, "job_id"])) & (js["desc_level"] != "")]
print("analysis set rows now: Spark", int((J2["skill"] == "Spark").sum()), "| Hadoop/Hive", int((J2["skill"] == "Hadoop/Hive").sum()))

SENT = re.compile(r"(?<=[.;!?])\s+")
GRAD = re.compile(r"\bbachelor[\u2019']?s?\b|\bgraduat(?:e|es|ion)\b|\bundergraduate\b|(?<!360-)(?<!360 )\bdegree\b(?!\s+of\b)|\bdiploma\b|\bb\.?\s?tech\b|\bbca\b|\bb\.?\s?sc\b|\bb\.?\s?com\b|\bbba\b|\bB\.\s?E\b\.?", re.I)
MASTER = re.compile(r"\bmaster[\u2019']?s\b|\bmasters? degree\b|\bmaster of\b|\bpost[- ]?graduate\b|\bmba\b|\bpgdm\b|\bm\.?\s?tech\b|\bmca\b|\bm\.?\s?sc\b|\bph\.?\s?d\b|\bdoctorate\b", re.I)
NOT_REQ = re.compile(r"\bdegree (?:is )?not (?:required|mandatory)\b|\bno degree\b|\bgraduation not required\b|\bwithout a degree\b|\bopen to all backgrounds\b|\bany background\b", re.I)
PREFC = re.compile(r"\bpreferred\b|\bplus\b|\badvantage\b|\bdesirable\b|\bnice to have\b|\bgood to have\b|\ba bonus\b|\ban asset\b|\bbeneficial\b", re.I)
RANKE = {"": 0, "not_required": 1, "bachelor": 2, "master": 3}

def edu_level(text):
    best, trig, pref_seen = "", "", False
    for line in text.split("\n"):
        for sent in SENT.split(line):
            s = re.sub(r"preferred candidate profile", "", sent, flags=re.I)
            has_m, has_g, has_nr = bool(MASTER.search(s)), bool(GRAD.search(s)), bool(NOT_REQ.search(s))
            if not (has_m or has_g or has_nr):
                continue
            if PREFC.search(s):
                pref_seen = True
                continue
            lvl = "not_required" if has_nr else ("master" if (has_m and not has_g) else "bachelor")
            if RANKE[lvl] > RANKE[best]:
                best, trig = lvl, sent
    if best == "" and pref_seen:
        best = "preferred_only"
    return best, re.sub(r"\s+", " ", trig)[:130], pref_seen

res = df["description_clean"].map(edu_level)
df["edu_level_desc"] = res.map(lambda x: x[0])
df["edu_degree_pref_seen"] = res.map(lambda x: x[2])
trig = dict(zip(df["job_id"], res.map(lambda x: x[1])))

lv = df.loc[ana, "edu_level_desc"].replace("", "none")
print("education level from description (analysis set):", lv.value_counts().to_dict())
print(pd.crosstab(df.loc[ana, "source"], lv).to_string())

print("--- master as the minimum")
for jid in df.loc[ana & (df["edu_level_desc"] == "master"), "job_id"]:
    print(jid, repr(trig[jid]))
print("--- not required")
for jid in df.loc[ana & (df["edu_level_desc"] == "not_required"), "job_id"]:
    print(jid, repr(trig[jid]))
print("--- 10 random bachelor decisions")
smp = df[ana & (df["edu_level_desc"] == "bachelor")].sample(10, random_state=4)
for jid in smp["job_id"]:
    print(jid, repr(trig[jid]))

def portal_edu(s):
    if s in ("Not mentioned", ""):
        return ""
    t = s.lower()
    if "graduation not required" in t:
        return "not_required"
    if "any graduate" in t:
        return "any_graduate"
    return "specific"

df["edu_portal_class"] = df["education"].map(portal_edu)
has_p = ana & (df["edu_portal_class"] != "")
print("portal education class (analysis set):", df.loc[ana, "edu_portal_class"].replace("", "none").value_counts().to_dict())
print(pd.crosstab(df.loc[has_p, "edu_portal_class"], df.loc[has_p, "edu_level_desc"].replace("", "none")).to_string())

c_all = df["edu_portal_class"].isin(["any_graduate", "not_required"]) & (df["edu_level_desc"] == "master")
c_all |= (df["edu_portal_class"] == "not_required") & df["edu_level_desc"].isin(["bachelor", "master"])
df["edu_conflict_flag"] = c_all
print("portal vs description conflicts, all rows:", int(c_all.sum()), "| analysis set:", int((c_all & ana).sum()))
for jid, pc, dl, a in zip(df.loc[c_all, "job_id"], df.loc[c_all, "edu_portal_class"], df.loc[c_all, "edu_level_desc"], ana[c_all]):
    print("  ", jid, pc, "->", dl, "| analysis set:", bool(a))

df.drop(columns=["_cands", "_excl"], errors="ignore").to_csv("../data_clean/stage13_education.csv", index=False, encoding="utf-8")
print("saved", df.shape)

In [ ]:
old_edu = df["edu_level_desc"].copy()

ABBR = re.compile(r"\b([BM])\.\s?(Sc|Tech|Com|A|E|Ed|Phil)\b\.?|\bPh\.\s?D\b\.?", re.I)
def squash_abbr(s):
    return ABBR.sub(lambda m: (m.group(1) + m.group(2)) if m.group(1) else "PhD", s)

GRAD_W = re.compile(r"\bbachelor[\u2019']?s?\b|\bgraduat(?:e|es|ion)\b|\bundergraduate\b|(?<!360-)(?<!360 )\bdegree\b(?!\s+of\b)|\bdiploma\b", re.I)
GRAD_A_CI = re.compile(r"\b(?:BTech|BSc|BCom|BCA|BBA)\b", re.I)
GRAD_A_CS = re.compile(r"\b(?:BE|BA|BMS)\b")
MASTER_W = re.compile(r"\bmaster[\u2019']s\b|\bmasters? degree\b|\bmaster of\b|\bpost[- ]?graduate\b|\bdoctorate\b", re.I)
MASTER_A = re.compile(r"\b(?:MBA|PGDM|MCA|MSc|MTech|PhD)\b", re.I)
AUD = re.compile(r"\b(?:fresh|recent|new|freshly)\s+(?:graduates?|grads?)\b|\bpass[- ]?outs?\b", re.I)
RANK2 = {"bachelor": 1, "master": 2}

def sent_level(s):
    s = squash_abbr(re.sub(r"preferred candidate profile", "", s, flags=re.I))
    if NOT_REQ.search(s):
        return "not_required"
    has_m = bool(MASTER_W.search(s) or MASTER_A.search(s))
    s2 = MASTER_W.sub(" ", s)
    s3 = AUD.sub(" ", s2)
    if GRAD_W.search(s3) or GRAD_A_CI.search(s3) or GRAD_A_CS.search(s3):
        return "bachelor"
    if has_m:
        return "master"
    if AUD.search(s2) or GRAD_W.search(s2):
        return "graduates_welcome"
    return ""

def edu_level(text):
    text = squash_abbr(text)
    levels, trig, pref_seen = [], {}, False
    for line in text.split("\n"):
        for sent in SENT.split(line):
            lvl = sent_level(sent)
            if not lvl:
                continue
            if PREFC.search(sent):
                pref_seen = True
                continue
            levels.append(lvl)
            trig.setdefault(lvl, re.sub(r"\s+", " ", sent)[:130])
    explicit = [l for l in levels if l in RANK2]
    if explicit:
        best = min(explicit, key=lambda x: RANK2[x])
    elif "not_required" in levels:
        best = "not_required"
    elif "graduates_welcome" in levels:
        best = "graduates_welcome"
    elif pref_seen:
        best = "preferred_only"
    else:
        best = ""
    return best, trig.get(best, ""), pref_seen

res = df["description_clean"].map(edu_level)
df["edu_level_desc"] = res.map(lambda x: x[0])
df["edu_degree_pref_seen"] = res.map(lambda x: x[2])
trig = dict(zip(df["job_id"], res.map(lambda x: x[1])))

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
lv = df.loc[ana, "edu_level_desc"].replace("", "none")
print("education level from description (analysis set):", lv.value_counts().to_dict())
print(pd.crosstab(df.loc[ana, "source"], lv).to_string())

ch = df["edu_level_desc"] != old_edu
print("--- rows whose level changed:", int(ch.sum()), "| in analysis set:", int((ch & ana).sum()))
for jid, o, n, a in zip(df.loc[ch, "job_id"], old_edu[ch].replace("", "none"), df.loc[ch, "edu_level_desc"].replace("", "none"), ana[ch]):
    print("  ", jid, o, "->", n, "| analysis set:", bool(a))

print("--- master as the minimum (analysis set)")
for jid in df.loc[ana & (df["edu_level_desc"] == "master"), "job_id"]:
    print(jid, repr(trig[jid]))
print("--- graduates_welcome (analysis set), first 8")
for jid in df.loc[ana & (df["edu_level_desc"] == "graduates_welcome"), "job_id"].head(8):
    print(jid, repr(trig[jid]))
print("--- not required")
for jid in df.loc[ana & (df["edu_level_desc"] == "not_required"), "job_id"]:
    print(jid, repr(trig[jid]))

print("--- job 160, lines with 'graduat'")
t160 = df.loc[df["job_id"] == "160", "description_clean"].iloc[0]
for m in list(re.finditer(r"graduat", t160, flags=re.I))[:4]:
    print(repr(re.sub(r"\s+", " ", t160[max(0, m.start() - 60): m.end() + 80])))

has_p = ana & (df["edu_portal_class"] != "")
print("portal education class (analysis set):", df.loc[ana, "edu_portal_class"].replace("", "none").value_counts().to_dict())
print(pd.crosstab(df.loc[has_p, "edu_portal_class"], df.loc[has_p, "edu_level_desc"].replace("", "none")).to_string())
c_all = (df["edu_portal_class"].isin(["any_graduate", "not_required"]) & (df["edu_level_desc"] == "master"))
c_all |= (df["edu_portal_class"] == "not_required") & df["edu_level_desc"].isin(["bachelor", "master"])
df["edu_conflict_flag"] = c_all
print("portal vs description conflicts, all rows:", int(c_all.sum()), "| analysis set:", int((c_all & ana).sum()))
for jid, pc, dl, a in zip(df.loc[c_all, "job_id"], df.loc[c_all, "edu_portal_class"], df.loc[c_all, "edu_level_desc"], ana[c_all]):
    print("  ", jid, pc, "->", dl, "| analysis set:", bool(a))

df.drop(columns=["_cands", "_excl"], errors="ignore").to_csv("../data_clean/stage13_education.csv", index=False, encoding="utf-8")
print("saved", df.shape)

In [ ]:
ana = ~df["off_target"] & ~df["is_repost_or_dup"]

def pref_sentences(text):
    text = squash_abbr(text)
    out = []
    for line in text.split("\n"):
        for sent in SENT.split(line):
            if sent_level(sent) and PREFC.search(sent):
                out.append(re.sub(r"\s+", " ", sent)[:150])
    return out

print("--- unread changes: level and trigger sentence")
for jid in ["11", "44", "140"]:
    lvl = df.loc[df["job_id"] == jid, "edu_level_desc"].iloc[0]
    print(jid, lvl, "|", repr(trig[jid]))
    for s in pref_sentences(df.loc[df["job_id"] == jid, "description_clean"].iloc[0])[:2]:
        print("     skipped as preferred:", repr(s))

po = df[ana & (df["edu_level_desc"] == "preferred_only")]
print("--- preferred_only rows (analysis set):", len(po))
for jid, tx in zip(po["job_id"], po["description_clean"]):
    ps = pref_sentences(tx)
    print(jid, repr(ps[0]) if ps else "(no sentence)")

In [ ]:
old_edu = df["edu_level_desc"].copy()

GRAD_A_CS = re.compile(r"\bBMS\b|\bBE(?=\s*/|\s*,|\s+or\b)")

def clean_pc(s):
    return re.sub(r"preferred candidate profile", "", s, flags=re.I)

def sent_level(s):
    s = squash_abbr(clean_pc(s))
    if NOT_REQ.search(s):
        return "not_required"
    has_m = bool(MASTER_W.search(s) or MASTER_A.search(s))
    s2 = MASTER_W.sub(" ", s)
    s3 = AUD.sub(" ", s2)
    if GRAD_W.search(s3) or GRAD_A_CI.search(s3) or GRAD_A_CS.search(s3):
        return "bachelor"
    if has_m:
        return "master"
    if AUD.search(s2) or GRAD_W.search(s2):
        return "graduates_welcome"
    return ""

def edu_level(text):
    text = squash_abbr(text)
    levels, trig, pref_seen = [], {}, False
    for line in text.split("\n"):
        for sent in SENT.split(line):
            lvl = sent_level(sent)
            if not lvl:
                continue
            if PREFC.search(clean_pc(sent)):
                pref_seen = True
                continue
            levels.append(lvl)
            trig.setdefault(lvl, re.sub(r"\s+", " ", sent)[:130])
    explicit = [l for l in levels if l in RANK2]
    if explicit:
        best = min(explicit, key=lambda x: RANK2[x])
    elif "not_required" in levels:
        best = "not_required"
    elif "graduates_welcome" in levels:
        best = "graduates_welcome"
    elif pref_seen:
        best = "preferred_only"
    else:
        best = ""
    return best, trig.get(best, ""), pref_seen

res = df["description_clean"].map(edu_level)
df["edu_level_desc"] = res.map(lambda x: x[0])
df["edu_degree_pref_seen"] = res.map(lambda x: x[2])
trig = dict(zip(df["job_id"], res.map(lambda x: x[1])))

MANUAL_BACHELOR = ["16", "87", "106", "117", "131", "200"]
df["edu_manual"] = df["job_id"].isin(MANUAL_BACHELOR)
df.loc[df["edu_manual"], "edu_level_desc"] = "bachelor"

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
lv = df.loc[ana, "edu_level_desc"].replace("", "none")
print("education level (analysis set):", lv.value_counts().to_dict())
print(pd.crosstab(df.loc[ana, "source"], lv).to_string())

ch = df["edu_level_desc"] != old_edu
print("--- changed vs last run:", int(ch.sum()))
for jid, o, n, a in zip(df.loc[ch, "job_id"], old_edu[ch].replace("", "none"), df.loc[ch, "edu_level_desc"].replace("", "none"), ana[ch]):
    print("  ", jid, o, "->", n, "| analysis set:", bool(a))
print("--- preferred_only rows (analysis set):", df.loc[ana & (df["edu_level_desc"] == "preferred_only"), "job_id"].tolist())
for jid in ["11", "44", "140"]:
    print(jid, df.loc[df["job_id"] == jid, "edu_level_desc"].iloc[0], "|", repr(trig[jid]))

has_p = ana & (df["edu_portal_class"] != "")
print(pd.crosstab(df.loc[has_p, "edu_portal_class"], df.loc[has_p, "edu_level_desc"].replace("", "none")).to_string())
c_all = (df["edu_portal_class"].isin(["any_graduate", "not_required"]) & (df["edu_level_desc"] == "master"))
c_all |= (df["edu_portal_class"] == "not_required") & df["edu_level_desc"].isin(["bachelor", "master"])
df["edu_conflict_flag"] = c_all
print("portal vs description conflicts:", df.loc[c_all, "job_id"].tolist(), "| analysis set:", df.loc[c_all & ana, "job_id"].tolist())

print("--- gender wording in descriptions (all 300 rows)")
GEN = re.compile(r"\b(?:male|female|men|women|ladies|girls|boys)\b[^.\n]{0,25}\b(?:only|preferred|candidates?)\b|\bonly (?:male|female|men|women|ladies|girls)\b|\bgender\s*:|\bmale\s*/\s*female\b|\bmale or female\b", re.I)
k = 0
for jid, src, tx, a, i in zip(df["job_id"], df["source"], df["description_clean"], ana, df["is_internship"]):
    m = GEN.search(tx)
    if m:
        k += 1
        if k <= 30:
            print(jid, src[:3], "| analysis set:", bool(a), "|", repr(re.sub(r"\s+", " ", tx[max(0, m.start() - 40): m.end() + 40])))
print("rows with gender wording:", k)

df.drop(columns=["_cands", "_excl"], errors="ignore").to_csv("../data_clean/stage13_education.csv", index=False, encoding="utf-8")
print("saved", df.shape)

In [ ]:
for jid, reason in [("11", "male candidates only"), ("67", "age limit 20 to 30 years")]:
    i = df["job_id"] == jid
    df.loc[i, "eligibility_restricted"] = True
    df.loc[i, "eligibility_reason"] = reason

ana = ~df["off_target"] & ~df["is_repost_or_dup"]
print("eligibility_restricted: all rows", int(df["eligibility_restricted"].sum()),
      "| analysis set", int((df["eligibility_restricted"] & ana).sum()))
print("reasons:", df.loc[df["eligibility_restricted"], ["job_id", "eligibility_reason"]].values.tolist())

AGE = re.compile(r"\bage\s*(?:limit|range|group|criteria|bar)?\s*[:\-]?\s*(?:below |under |upto |up to |max(?:imum)? |between )?\d{2}\b|\b\d{2}\s*(?:to|-|\u2013)\s*\d{2}\s*years\s*(?:old|of age)|\byears of age\b|\bage below\b|\bunder the age\b|\bmaximum age\b", re.I)
k = 0
print("--- age wording in descriptions (all 300 rows)")
for jid, src, tx, a in zip(df["job_id"], df["source"], df["description_clean"], ana):
    m = AGE.search(tx)
    if m:
        k += 1
        if k <= 20:
            print(jid, src[:3], "| analysis set:", bool(a), "|", repr(re.sub(r"\s+", " ", tx[max(0, m.start() - 40): m.end() + 40])))
print("rows with age wording:", k)

df.drop(columns=["_cands", "_excl"], errors="ignore").to_csv("../data_clean/stage13_education.csv", index=False, encoding="utf-8")
print("saved", df.shape)

In [ ]:
import json
df["in_analysis_set"] = ~df["off_target"] & ~df["is_repost_or_dup"]

JOBS_COLS = [
 "job_id", "source", "job_title", "company", "city", "search_city", "primary_city", "city_flag",
 "work_mode_clean", "posted_ago", "age_days", "age_source", "is_old", "is_reposted_label", "refreshed_label_conflict",
 "experience_portal", "exp_min", "exp_max", "exp_form", "exp_source", "exp_phrases_disagree", "exp_lost_dash",
 "exp_portal_min", "exp_portal_max", "exp_min_fallback", "exp_fallback_source", "exp_conflict_flag",
 "desc_min_above_portal_max", "exp_band_desc", "exp_band_fallback", "exp_entry_wording",
 "entry_title_words", "entry_title_assoc", "entry_title_level", "entry_title_any", "entry_title_no_assoc", "entry_title_mixed",
 "da_title", "q1_base", "q1_assoc_sens",
 "salary_portal", "salary_min_lpa", "salary_max_lpa", "salary_mid_lpa", "salary_form", "salary_source",
 "salary_outlier", "salary_outlier_reason", "salary_usable", "salary_usable_clean",
 "edu_level_desc", "edu_portal_class", "edu_conflict_flag", "edu_manual",
 "is_internship", "is_remote", "is_contract", "has_bond_or_service_agreement",
 "eligibility_restricted", "eligibility_reason", "is_staffing_or_aggregator", "staffing_reason",
 "off_target", "off_target_reason", "borderline", "borderline_reason", "thin_description", "desc_len",
 "is_repost_or_dup", "dup_group", "dup_kept_job_id", "dup_confidence", "dup_raw_cities", "dup_sources",
 "excel_advanced", "in_analysis_set",
]
missing = [c for c in JOBS_COLS if c not in df.columns]
print("missing columns:", missing)
JOBS_COLS = [c for c in JOBS_COLS if c in df.columns]

jobs = df[JOBS_COLS].copy()
FLAG_COLS = [c for c in JOBS_COLS if c in (
 "is_old", "is_reposted_label", "refreshed_label_conflict", "exp_phrases_disagree", "exp_lost_dash",
 "exp_conflict_flag", "desc_min_above_portal_max", "exp_entry_wording", "entry_title_words", "entry_title_assoc",
 "entry_title_level", "entry_title_any", "entry_title_no_assoc", "entry_title_mixed", "da_title", "q1_base",
 "q1_assoc_sens", "salary_outlier", "salary_usable", "salary_usable_clean", "edu_conflict_flag", "edu_manual",
 "is_internship", "is_remote", "is_contract", "has_bond_or_service_agreement", "eligibility_restricted",
 "is_staffing_or_aggregator", "off_target", "borderline", "thin_description", "is_repost_or_dup",
 "excel_advanced", "in_analysis_set")]
NUM_COLS = [c for c in JOBS_COLS if c in ("age_days", "exp_min", "exp_max", "exp_portal_min", "exp_portal_max",
 "exp_min_fallback", "salary_min_lpa", "salary_max_lpa", "salary_mid_lpa", "desc_len")]
for c in FLAG_COLS:
    jobs[c] = jobs[c].map({True: 1, False: 0, "True": 1, "False": 0}).astype("Int64")
for c in NUM_COLS:
    jobs[c] = pd.to_numeric(jobs[c], errors="coerce")
jobs["job_id"] = jobs["job_id"].astype(int)
jobs["dup_group"] = pd.to_numeric(jobs["dup_group"], errors="coerce").astype("Int64")
jobs["dup_kept_job_id"] = pd.to_numeric(jobs["dup_kept_job_id"], errors="coerce").astype("Int64")
for c in jobs.columns:
    if jobs[c].dtype == object:
        jobs[c] = jobs[c].map(lambda v: None if (v is None or (isinstance(v, float) and pd.isna(v)) or str(v).strip() == "") else str(v).strip())

skills = pd.DataFrame({"skill": list(VOC), "skill_type": [type_map[s] for s in VOC]})
skills.insert(0, "skill_id", range(1, len(skills) + 1))
jsx = js.merge(skills[["skill_id", "skill"]], on="skill", how="left")
job_skills = jsx[["job_id", "skill_id", "desc_level", "desc_mentions", "in_portal_tags"]].copy()
job_skills["job_id"] = job_skills["job_id"].astype(int)
job_skills["desc_level"] = job_skills["desc_level"].replace("", None)
job_skills["desc_mentions"] = job_skills["desc_mentions"].astype(int)
job_skills["in_portal_tags"] = job_skills["in_portal_tags"].astype(int)

print("jobs:", jobs.shape, "| unique job_id:", jobs["job_id"].is_unique)
print("skills:", skills.shape, "| job_skills:", job_skills.shape)
print("job_skills job_ids not in jobs:", int((~job_skills["job_id"].isin(jobs["job_id"])).sum()),
      "| skill_ids missing:", int(job_skills["skill_id"].isna().sum()),
      "| duplicate (job, skill) pairs:", int(job_skills.duplicated(["job_id", "skill_id"]).sum()),
      "| rows with no description level and no tag:", int(((job_skills["desc_level"].isna()) & (job_skills["in_portal_tags"] == 0)).sum()))
print("NULL counts, columns with any NULL (first 25):")
nn = jobs.isna().sum()
print(nn[nn > 0].sort_values(ascending=False).head(25).to_dict())

a = jobs["in_analysis_set"] == 1
jsa = job_skills.merge(skills, on="skill_id").merge(jobs[["job_id", "in_analysis_set"]], on="job_id")
jsa = jsa[(jsa["in_analysis_set"] == 1) & jsa["desc_level"].notna()]
control = {
 "jobs_rows": int(len(jobs)),
 "analysis_set": int(a.sum()),
 "analysis_by_source": jobs.loc[a, "source"].value_counts().to_dict(),
 "analysis_by_city": jobs.loc[a, "primary_city"].value_counts().to_dict(),
 "off_target": int(jobs["off_target"].sum()),
 "dups": int(jobs["is_repost_or_dup"].sum()),
 "q1_base": int(jobs["q1_base"].sum()),
 "entry_any_in_analysis": int(((jobs["entry_title_any"] == 1) & a).sum()),
 "salary_usable": int(jobs["salary_usable"].sum()),
 "salary_usable_clean": int(jobs["salary_usable_clean"].sum()),
 "eligibility_restricted": int(jobs["eligibility_restricted"].sum()),
 "job_skills_rows": int(len(job_skills)),
 "skill_rows_in_analysis_with_desc": int(len(jsa)),
 "SQL_desc_analysis": int((jsa["skill"] == "SQL").sum()),
 "Excel_desc_analysis": int((jsa["skill"] == "Excel").sum()),
 "Python_desc_analysis": int((jsa["skill"] == "Python").sum()),
 "Spark_desc_analysis": int((jsa["skill"] == "Spark").sum()),
}
print(json.dumps(control, indent=1))

jobs.to_csv("../data_clean/stage14_jobs.csv", index=False, encoding="utf-8")
skills.to_csv("../data_clean/stage14_skills.csv", index=False, encoding="utf-8")
job_skills.to_csv("../data_clean/stage14_job_skills.csv", index=False, encoding="utf-8")
print("saved", jobs.shape, skills.shape, job_skills.shape)

In [ ]:
%pip install sqlalchemy pymysql python-dotenv 

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

print("env file found:", load_dotenv("../.env"))
url = URL.create("mysql+pymysql", username=os.getenv("MYSQL_USER"), password=os.getenv("MYSQL_PASSWORD"),
                 host=os.getenv("MYSQL_HOST", "localhost"), port=int(os.getenv("MYSQL_PORT", "3306")),
                 database=os.getenv("MYSQL_DB"), query={"charset": "utf8mb4"})
engine = create_engine(url)
print("password loaded:", bool(os.getenv("MYSQL_PASSWORD")))
with engine.connect() as conn:
    print("server:", conn.execute(text("SELECT VERSION()")).scalar())
    print("database:", conn.execute(text("SELECT DATABASE()")).scalar())
    for t in ["jobs", "skills", "job_skills"]:
        print(t, "rows:", conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar())

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv("../.env")
url = URL.create("mysql+pymysql", username=os.getenv("MYSQL_USER"), password=os.getenv("MYSQL_PASSWORD"),
                 host=os.getenv("MYSQL_HOST", "localhost"), port=int(os.getenv("MYSQL_PORT", "3306")),
                 database=os.getenv("MYSQL_DB"), query={"charset": "utf8mb4"})
engine = create_engine(url)

FILES = {"skills": "../data_clean/stage14_skills.csv",
         "jobs": "../data_clean/stage14_jobs.csv",
         "job_skills": "../data_clean/stage14_job_skills.csv"}

with engine.connect() as conn:
    meta = pd.read_sql(text(
        "SELECT table_name AS t, column_name AS c, data_type AS d, character_maximum_length AS L, "
        "ordinal_position AS o FROM information_schema.columns WHERE table_schema = 'job_market'"), conn)

ok = True
frames = {}
for t, path in FILES.items():
    d = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[""])
    m = meta[meta["t"] == t].sort_values("o")
    db_cols = m["c"].tolist()
    only_csv, only_db = sorted(set(d.columns) - set(db_cols)), sorted(set(db_cols) - set(d.columns))
    print(t, "| csv columns", d.shape[1], "| db columns", len(db_cols), "| only in csv:", only_csv, "| only in db:", only_db)
    if only_csv or only_db:
        ok = False
        continue
    d = d[db_cols]
    for _, r in m.iterrows():
        c, dt, L = r["c"], r["d"], r["L"]
        if dt in ("int", "tinyint", "smallint", "bigint"):
            before = int(d[c].notna().sum())
            d[c] = pd.to_numeric(d[c], errors="coerce").astype("Int64")
            if int(d[c].notna().sum()) != before:
                print("  NUMBER LOST in", t, c); ok = False
        elif dt == "decimal":
            before = int(d[c].notna().sum())
            d[c] = pd.to_numeric(d[c], errors="coerce")
            if int(d[c].notna().sum()) != before:
                print("  NUMBER LOST in", t, c); ok = False
        elif dt == "varchar":
            mx = d[c].dropna().str.len().max()
            if pd.notna(mx) and mx > L:
                print("  TOO LONG", t, c, "max", int(mx), "limit", int(L)); ok = False
    frames[t] = d
print("preflight ok:", ok)

if ok:
    def to_none(d):
        return d.astype(object).where(d.notna(), None)
    with engine.begin() as conn:
        for t in ["job_skills", "jobs", "skills"]:
            conn.execute(text(f"DELETE FROM {t}"))
        for t in ["skills", "jobs", "job_skills"]:
            to_none(frames[t]).to_sql(t, conn, if_exists="append", index=False, method="multi", chunksize=100)
            print("loaded", t, len(frames[t]))

    JS = ("FROM job_skills js JOIN jobs j ON j.job_id = js.job_id JOIN skills s ON s.skill_id = js.skill_id "
          "WHERE j.in_analysis_set = 1 AND js.desc_level IS NOT NULL")
    q = {
     "jobs_rows": ("SELECT COUNT(*) FROM jobs", 300),
     "skills_rows": ("SELECT COUNT(*) FROM skills", 45),
     "job_skills_rows": ("SELECT COUNT(*) FROM job_skills", 2007),
     "analysis_set": ("SELECT SUM(in_analysis_set) FROM jobs", 246),
     "off_target": ("SELECT SUM(off_target) FROM jobs", 32),
     "is_repost_or_dup": ("SELECT SUM(is_repost_or_dup) FROM jobs", 24),
     "q1_base": ("SELECT SUM(q1_base) FROM jobs", 102),
     "entry_any_in_analysis": ("SELECT COUNT(*) FROM jobs WHERE in_analysis_set = 1 AND entry_title_any = 1", 51),
     "salary_usable": ("SELECT SUM(salary_usable) FROM jobs", 51),
     "salary_usable_clean": ("SELECT SUM(salary_usable_clean) FROM jobs", 39),
     "eligibility_restricted": ("SELECT SUM(eligibility_restricted) FROM jobs", 13),
     "exp_min filled": ("SELECT COUNT(exp_min) FROM jobs", 178),
     "salary_form filled": ("SELECT COUNT(salary_form) FROM jobs", 70),
     "dup_group filled": ("SELECT COUNT(dup_group) FROM jobs", 44),
     "off_target_reason filled": ("SELECT COUNT(off_target_reason) FROM jobs", 32),
     "primary_city NULL": ("SELECT COUNT(*) FROM jobs WHERE primary_city IS NULL", 0),
     "skill rows, analysis set, with description level": ("SELECT COUNT(*) " + JS, 1574),
     "SQL (analysis set, description)": ("SELECT COUNT(*) " + JS + " AND s.skill = 'SQL'", 192),
     "Excel (analysis set, description)": ("SELECT COUNT(*) " + JS + " AND s.skill = 'Excel'", 150),
     "Python (analysis set, description)": ("SELECT COUNT(*) " + JS + " AND s.skill = 'Python'", 99),
     "Spark (analysis set, description)": ("SELECT COUNT(*) " + JS + " AND s.skill = 'Spark'", 9),
    }
    bad = []
    with engine.connect() as conn:
        print(f"{'check':52s} {'expected':>9s} {'mysql':>7s}")
        for k, (sql, exp) in q.items():
            v = conn.execute(text(sql)).scalar()
            v = int(v) if v is not None else 0
            flag = "OK" if v == exp else "MISMATCH"
            if v != exp:
                bad.append(k)
            print(f"{k:52s} {exp:9d} {v:7d}  {flag}")
        by_src = dict(conn.execute(text("SELECT source, COUNT(*) FROM jobs WHERE in_analysis_set = 1 GROUP BY source")).fetchall())
        by_city = dict(conn.execute(text("SELECT primary_city, COUNT(*) FROM jobs WHERE in_analysis_set = 1 GROUP BY primary_city")).fetchall())
    exp_src = {"Naukri": 76, "LinkedIn": 65, "Indeed": 53, "Foundit": 52}
    exp_city = {"Bengaluru": 55, "Delhi/NCR": 44, "Pune": 40, "Chennai": 36, "Hyderabad": 35, "Mumbai": 34, "Unknown": 1, "Other": 1}
    by_src = {k: int(v) for k, v in by_src.items()}
    by_city = {k: int(v) for k, v in by_city.items()}
    print("analysis set by source:", by_src, "| matches Pandas:", by_src == exp_src)
    print("analysis set by city:", by_city, "| matches Pandas:", by_city == exp_city)
    if by_src != exp_src or by_city != exp_city:
        bad.append("source/city split")
    print("RESULT:", "ALL CHECKS MATCH" if not bad else "MISMATCHES: " + ", ".join(bad))